# VECTRA: reproducible pipeline (Google Colab, free tier, CPU)

This notebook reproduces every number, table and figure of the manuscript. It downloads the open data from the Hugging Face Hub, trains all forecasters with five seeds, evaluates calibration, allocation and deployment, runs the statistical tests and draws the figures.

* Full run (`QUICK = False`): about 25 minutes on the 2-core CPU used for the paper; allow 30 to 60 minutes on a free Colab CPU.
* Smoke test (`QUICK = True`): two seeds and 100 scenario paths, roughly a third of the full run.

No GPU is needed. Outputs are written to the working folder as `results_*.json`, `fig*.png` and `cache_*.pt`.

In [ ]:
!pip -q install lightgbm pyarrow
import torch, sklearn, scipy, numpy, pandas
print(torch.__version__, sklearn.__version__, scipy.__version__, numpy.__version__, pandas.__version__)

## 1. Source modules
Each cell writes one module of the repository to the working folder.

In [ ]:
%%writefile vectra_data.py
"""Data access and preprocessing for VECTRA.

Primary data: weekly probable dengue cases (SINAN) and station climate (INMET) for 43
Brazilian mesoregions, 2019-2023, harmonised and hosted on the Hugging Face Hub
(dataset thiagorfreitas/dengue-tcc2-data, folder data/model_ready/mesorregiao).
External data: InfoDengue weekly series for Belem (PA), hosted at JeanDias/InfoDengue_Belem.
"""
import os, urllib.request
import numpy as np, pandas as pd

HF = "https://huggingface.co/datasets"
MESO_FILES = {s: f"{HF}/thiagorfreitas/dengue-tcc2-data/resolve/main/data/model_ready/mesorregiao/{s}.parquet"
              for s in ("train", "val", "test")}
BELEM_FILE = f"{HF}/JeanDias/InfoDengue_Belem/resolve/main/dengue_34-1.csv"
STATE_OF = {"31": "MG", "32": "ES", "33": "RJ", "35": "SP", "52": "GO", "53": "DF"}


def _get(url, path):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


def _fill_climate(X, woy, train_mask, groups):
    """X: [T, N] climate matrix with NaN. Fill short gaps by interpolation (<=3 weeks),
    then by same-state mean of the week, then by training-period week-of-year climatology."""
    df = pd.DataFrame(X)
    df = df.interpolate(limit=3, limit_area="inside", axis=0)
    X = df.values.copy()
    for g in np.unique(groups):
        cols = np.where(groups == g)[0]
        m = np.nanmean(X[:, cols], axis=1) if len(cols) > 1 else np.full(X.shape[0], np.nan)
        for c in cols:
            miss = np.isnan(X[:, c])
            X[miss, c] = m[miss]
    clim = np.full((54, X.shape[1]), np.nan)
    for w in range(1, 54):
        sel = (woy == w) & train_mask
        if sel.any():
            clim[w] = np.nanmean(X[sel], axis=0)
    clim = pd.DataFrame(clim).interpolate(axis=0, limit_direction="both").values
    glob = np.nanmean(clim, axis=1, keepdims=True)
    clim = np.where(np.isnan(clim), glob, clim)
    for t in range(X.shape[0]):
        miss = np.isnan(X[t])
        X[t, miss] = clim[woy[t], miss]
    return X, clim


def load_meso(cache="."):
    parts = [pd.read_parquet(_get(u, f"{cache}/data_meso_{s}.parquet")) for s, u in MESO_FILES.items()]
    d = pd.concat(parts, ignore_index=True)
    d["unit"] = d["ibge_municipio"].astype(str)
    d = d.sort_values(["ano", "semana_epidemiologica", "unit"])
    # the final epidemiological week of 2023 is a partial stub (incomplete reporting) and is excluded
    d = d[~((d.ano == 2023) & (d.semana_epidemiologica == 53))]
    units = sorted(d.unit.unique())
    base_idx = d.pivot_table(index=["ano", "semana_epidemiologica"], columns="unit",
                             values="qt_confirmados_provaveis", aggfunc="first").index
    piv = lambda c: d.pivot_table(index=["ano", "semana_epidemiologica"], columns="unit", values=c,
                                  aggfunc="first", dropna=False).reindex(index=base_idx, columns=units)
    y = piv("qt_confirmados_provaveis").values.astype(np.float64)
    idx = piv("qt_confirmados_provaveis").index
    year = np.array([a for a, _ in idx]); woy = np.array([w for _, w in idx])
    train_mask = year <= 2021
    groups = np.array([STATE_OF[u[:2]] for u in units])
    clim = {}
    for name, col in [("temp", "temp_mean_c"), ("hum", "humidity_mean_pct"), ("rain", "rain_sum_mm")]:
        raw = piv(col).values.astype(np.float64)
        miss = np.isnan(raw).mean()
        X, cl = _fill_climate(raw, woy, train_mask, groups)
        clim[name] = X; clim[name + "_clim"] = cl; clim[name + "_missing"] = float(miss)
    delay = np.clip(np.nan_to_num(piv("atraso_notificacao_medio_dias").values.astype(np.float64)), 0, 60)
    return dict(name="SE-Brazil", y=y, year=year, woy=woy, units=units, groups=groups,
                temp=clim["temp"], hum=clim["hum"], rain=clim["rain"], delay=delay,
                temp_clim=clim["temp_clim"], hum_clim=clim["hum_clim"], rain_clim=clim["rain_clim"],
                missing={k: clim[k + "_missing"] for k in ("temp", "hum", "rain")},
                split=dict(train_end=int(np.where(year <= 2021)[0].max()),
                           val_end=int(np.where(year <= 2022)[0].max()), end=len(year) - 1))


def load_belem(cache="."):
    b = pd.read_csv(_get(BELEM_FILE, f"{cache}/data_belem.csv")).sort_values("SE")
    b = b[(b.SE >= 201001) & (b.SE <= 202452)].reset_index(drop=True)
    year = (b.SE // 100).values; woy = (b.SE % 100).values
    y = b["casos"].values.astype(np.float64)[:, None]
    train_mask = year <= 2020
    groups = np.array(["PA"])
    temp, tc = _fill_climate(b["tempmed"].values[:, None].astype(float), woy, train_mask, groups)
    hum, hc = _fill_climate(b["umidmed"].values[:, None].astype(float), woy, train_mask, groups)
    rain = np.zeros_like(temp); rc = np.zeros((54, 1))
    return dict(name="Belem", y=y, year=year, woy=woy, units=["Belem"], groups=groups,
                temp=temp, hum=hum, rain=rain, delay=np.zeros_like(temp),
                temp_clim=tc, hum_clim=hc, rain_clim=rc,
                missing={"temp": float(b["tempmed"].isna().mean()), "hum": float(b["umidmed"].isna().mean()), "rain": 1.0},
                split=dict(train_end=int(np.where(year <= 2020)[0].max()),
                           val_end=int(np.where(year <= 2021)[0].max()), end=len(year) - 1))


def coupling_graph(D, k=5):
    """Inter-regional coupling estimated from training-period co-movement of weekly log-growth.
    Returns a row-normalised kNN matrix with zero diagonal (no mobility data are assumed)."""
    y = D["y"][: D["split"]["train_end"] + 1]
    N = y.shape[1]
    if N == 1:
        return np.zeros((1, 1))
    g = np.diff(np.log1p(y), axis=0)
    C = np.corrcoef(g.T); np.fill_diagonal(C, -np.inf)
    A = np.zeros((N, N))
    for i in range(N):
        nb = np.argsort(C[i])[::-1][:k]
        w = np.clip(C[i, nb], 1e-3, None)
        A[i, nb] = w / w.sum()
    return A


In [ ]:
%%writefile vectra_mech.py
"""Mechanistic components of VECTRA: thermal suitability, temperature-dependent
extrinsic incubation and generation interval, and intervention effects on the
reproduction number.

Anchors taken from the literature (not fitted here):
  * Ae. aegypti dengue R0 thermal limits/optimum: 17.8, 29.1, 34.6 degC (Mordecai et al., 2017).
  * Mean EIP 15 d at 25 degC and 6.5 d at 30 degC (Chan & Johansson, 2012) -> log-linear EIP(T).
  * Mean intrinsic incubation 5.9 d (Chan & Johansson, 2012).
Assumptions (varied in the sensitivity analysis): host-to-vector and vector-to-host contact delay
C0 = 4 d, baseline adult mortality MU0 = 0.10 / d, intervention efficacies E_L, E_A, E_S.
"""
import numpy as np, torch

T0, TOPT, TM = 17.8, 29.1, 34.6
R_SHAPE = (TM - TOPT) / (TOPT - T0)
EIP25, EIP30 = 15.0, 6.5
KAPPA = np.log(EIP25 / EIP30) / 5.0
IIP, C0 = 5.9, 4.0
MU0 = 0.10
GI_CV = 0.5
K_GI = 8                       # renewal kernel support (weeks)
ACTIONS = ("larviciding", "adulticiding", "source_reduction")
EFF = dict(larviciding=0.30, adulticiding=0.20, source_reduction=0.20)
COST = dict(larviciding=1.0, adulticiding=1.5, source_reduction=0.8)


def log_suitability(T):
    """log of the Yan-Hunt beta-shaped thermal suitability with exponent 1 (peak value 0 at TOPT)."""
    lib = torch if torch.is_tensor(T) else np
    T = lib.clip(T, T0 + 0.3, TM - 0.3)
    return lib.log((T - T0) / (TOPT - T0)) + R_SHAPE * lib.log((TM - T) / (TM - TOPT))


def eip_days(T):
    lib = torch if torch.is_tensor(T) else np
    return lib.clip(EIP25 * lib.exp(-KAPPA * (T - 25.0)), 4.0, 40.0)


def gi_mean_weeks(T):
    return (IIP + eip_days(T) + C0) / 7.0


def gi_kernel(m):
    """Discretised gamma generation-interval kernel with mean m (weeks) and CV GI_CV,
    evaluated on lags 1..K_GI. Works for numpy arrays or torch tensors of any shape."""
    is_t = torch.is_tensor(m)
    lib = torch if is_t else np
    shape = 1.0 / GI_CV ** 2
    k = lib.arange(1, K_GI + 1, dtype=m.dtype if is_t else float)
    if is_t:
        k = k.to(m.device)
    scale = m[..., None] / shape
    # log-density of gamma at bin centres, normalised over the support
    if is_t:
        logp = (shape - 1) * torch.log(k) - k / scale - shape * torch.log(scale) - torch.lgamma(torch.tensor(shape))
        return torch.softmax(logp, dim=-1)
    from scipy.special import gammaln
    logp = (shape - 1) * np.log(k) - k / scale - shape * np.log(scale) - gammaln(shape)
    p = np.exp(logp - logp.max(-1, keepdims=True))
    return p / p.sum(-1, keepdims=True)


def phi(action, T, eff=None, mu0=MU0):
    """Reduction of log R under full coverage of an action at mean temperature T.
    Larviciding / source reduction scale adult density; adulticiding raises mortality, which
    lowers equilibrium density (1/mu) and survival through the EIP (exp(-mu*EIP))."""
    eff = EFF if eff is None else eff
    lib = torch if torch.is_tensor(T) else np
    e = eff[action]
    if action == "adulticiding":
        return 2.0 * np.log1p(e) + mu0 * e * eip_days(T)
    return -np.log1p(-e) + 0.0 * T


In [ ]:
%%writefile vectra_models.py
"""Forecasting models: VECTRA (mechanistic-neural renewal) and neural baselines."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import vectra_mech as M

L_HIST, H = 12, 8
QS = np.array([0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975])


# ----------------------------------------------------------------------------- tensors
class Window:
    """Builds per-origin inputs. Origin t = last observed week; targets t+1..t+H."""

    def __init__(self, D, A, feat_stats=None):
        self.D, self.A = D, A
        y = D["y"]; self.y = y
        Ay = y @ A.T if A.shape[0] > 1 else np.zeros_like(y)
        cum52 = np.array([y[max(0, t - 51): t + 1].sum(0) for t in range(len(y))])
        self.cum52 = cum52
        raw = np.stack([np.log1p(y), np.log1p(Ay), D["temp"], D["hum"], np.log1p(D["rain"]),
                        np.log1p(D["delay"]), np.log1p(cum52)], -1)        # [T,N,7]
        te = D["split"]["train_end"]
        if feat_stats is None:
            mu = raw[: te + 1].reshape(-1, raw.shape[-1]).mean(0)
            sd = raw[: te + 1].reshape(-1, raw.shape[-1]).std(0) + 1e-6
            feat_stats = (mu, sd)
        self.feat_stats = feat_stats
        z = (raw - feat_stats[0]) / feat_stats[1]
        ang = 2 * np.pi * D["woy"] / 52.18
        seas = np.stack([np.sin(ang), np.cos(ang)], -1)[:, None, :].repeat(y.shape[1], 1)
        self.X = np.concatenate([z, seas], -1).astype(np.float32)            # [T,N,9]
        # 4-week trailing mean temperature (observed) and the same quantity with climatology for unseen weeks
        self.temp = D["temp"]; self.clim = D["temp_clim"]

    def pool_init(self):
        te = self.D["split"]["train_end"]
        return 3.0 * self.cum52[: te + 1].max(0) + 100.0

    def tbar_obs(self, t, temp=None, n=3):
        temp = self.temp if temp is None else temp
        return np.stack([temp[s - 3: s + 1].mean(0) for s in range(t - n + 1, t + 1)], 1)

    def tbar_future(self, t, temp=None):
        """Trailing 4-week mean temperature for target weeks t+1..t+H; weeks after t use climatology."""
        temp = self.temp if temp is None else temp
        T, N = temp.shape
        out = np.zeros((N, H))
        for h in range(1, H + 1):
            ws = range(t + h - 4, t + h)
            vals = []
            for w in ws:
                if w <= t:
                    vals.append(temp[w])
                else:
                    woy = self.D["woy"][min(w, T - 1)] if w < T else ((self.D["woy"][t] + (w - t) - 1) % 52) + 1
                    vals.append(self.clim[woy])
            out[:, h - 1] = np.mean(vals, 0)
        return out

    def batch(self, origins, end, X=None, y=None, temp=None):
        X = self.X if X is None else X
        y = self.y if y is None else y
        xs, yh, tb, tgt, msk, cum = [], [], [], [], [], []
        for t in origins:
            cum.append(y[max(0, t - 51): t + 1].sum(0))
            xs.append(X[t - L_HIST + 1: t + 1].transpose(1, 0, 2))
            yh.append(y[t - L_HIST + 1: t + 1].T)
            tb.append(np.concatenate([self.tbar_obs(t, temp), self.tbar_future(t, temp)], 1))
            tg = np.zeros((y.shape[1], H)); mk = np.zeros((y.shape[1], H))
            for h in range(1, H + 1):
                if t + h <= end:
                    tg[:, h - 1] = self.y[t + h]; mk[:, h - 1] = 1
            tgt.append(tg); msk.append(mk)
        f = lambda a: torch.tensor(np.array(a), dtype=torch.float32)
        yh_t = f(yh)
        self.last_cum = f(cum)
        return f(xs), torch.cat([yh_t, self.last_cum[..., None]], -1), f(tb), f(tgt), f(msk)


def nb_nll(y, mu, r):
    mu = mu.clamp_min(1e-6)
    return -(torch.lgamma(y + r) - torch.lgamma(r) - torch.lgamma(y + 1)
             + r * torch.log(r / (r + mu)) + y * torch.log(mu / (r + mu)))


def nb_sample(mu, r, n):
    p = (mu / (mu + r)).clamp(1e-6, 1 - 1e-6)
    return torch.distributions.NegativeBinomial(total_count=r, probs=p).sample((n,))


class Encoder(nn.Module):
    def __init__(self, N, f_in=9, hid=32, emb=4):
        super().__init__()
        self.gru = nn.GRU(f_in, hid, batch_first=True)
        self.emb = nn.Embedding(N, emb)
        self.out_dim = hid + emb

    def forward(self, x):                      # x [B,N,L,F]
        B, N, L, Fd = x.shape
        _, h = self.gru(x.reshape(B * N, L, Fd))
        h = h[-1].reshape(B, N, -1)
        e = self.emb.weight[None].expand(B, -1, -1)
        return torch.cat([h, e], -1)


# ----------------------------------------------------------------------------- VECTRA
class VECTRA(nn.Module):
    """log R_{i,t+h} = b_i + gamma * log S(Tbar) + delta_{i,t,h};  lambda = R * sum_k w_k(Tbar) ymix_{t+h-k} + eta_i
    ymix = (1-omega) y_i + omega (A y)_i ; delta from a bounded neural closure."""

    def __init__(self, N, A, thermal=True, gi_temp=True, graph=True, closure=True, bound=2.0, deplete=False,
                 pool_init=None, obs_closure=True, obs_bound=1.0, n_rhat=3):
        super().__init__()
        self.n_rhat = n_rhat
        self.obs_closure, self.obs_bound = obs_closure and closure, obs_bound
        self._nu = None
        self.deplete = deplete
        self.p = nn.Parameter(torch.tensor(np.log(pool_init) if pool_init is not None else np.full(N, 12.0),
                                           dtype=torch.float32))
        self.N = N; self.thermal, self.gi_temp, self.graph, self.closure = thermal, gi_temp, graph and N > 1, closure
        self.register_buffer("A", torch.tensor(A, dtype=torch.float32))
        self.b = nn.Parameter(torch.zeros(N))
        self.g_raw = nn.Parameter(torch.tensor(0.0))
        self.w_raw = nn.Parameter(torch.tensor(-2.0))
        self.eta = nn.Parameter(torch.zeros(N))
        self.r_raw = nn.Parameter(torch.full((N,), 1.5))
        self.bound = bound
        self.rho_raw = nn.Parameter(torch.tensor(2.0))
        if closure:
            self.enc = Encoder(N)
            self.head = nn.Sequential(nn.Linear(self.enc.out_dim, 32), nn.GELU(), nn.Linear(32, 3 * H))
        self.dh = nn.Parameter(torch.zeros(H))

    @property
    def gamma(self):
        return F.softplus(self.g_raw) if self.thermal else torch.tensor(0.0)

    @property
    def omega(self):
        return torch.sigmoid(self.w_raw) if self.graph else torch.tensor(0.0)

    def disp(self):
        return self._r_full

    def log_r(self, x, yh, tbar):
        """Origin-anchored reproduction number propagated by temperature:
        log R_{t+h} = rho^h log Rhat_t + (1-rho^h) b_i + gamma [log S(Tbar_{t+h}) - log S(Tbar_t)] + delta_h."""
        y12 = yh[..., :-1]
        tb_obs, tb_fut = tbar[..., :3], tbar[..., 3:]
        w_obs = self.kernel(tb_obs)                                             # [B,N,3,K]
        mix = self.mix(y12)
        lam = []
        for j in range(3):                                                      # s = t-2, t-1, t
            s_idx = y12.shape[-1] - 3 + j
            hist = mix[..., s_idx - M.K_GI: s_idx]                              # lags K..1
            lam.append((hist.flip(-1) * w_obs[..., j, :]).sum(-1))
        lam_list = lam
        nr = self.n_rhat
        lam_n = torch.stack(lam_list[-nr:], -1).sum(-1)
        log_rhat = torch.log((y12[..., -nr:].sum(-1) + 1.0) / (lam_n + 1.0))
        hs = torch.arange(1, H + 1, dtype=tbar.dtype)
        rho = torch.sigmoid(self.rho_raw)
        decay = rho ** hs
        lr = decay * log_rhat[..., None] + (1 - decay) * self.b[None, :, None]
        lr = lr + self.gamma * (M.log_suitability(tb_fut) - M.log_suitability(tb_obs[..., -1:]))
        delta = torch.zeros_like(lr)
        kap = torch.zeros_like(lr)
        self._nu = torch.zeros_like(lr)
        if self.closure:
            o = self.head(self.enc(x))
            delta = self.bound * torch.tanh(o[..., :H] / self.bound)
            kap = 2.0 * torch.tanh(o[..., H:2 * H] / 2.0)
            self._nu = self.obs_bound * torch.tanh(o[..., 2 * H:] / self.obs_bound) if self.obs_closure else torch.zeros_like(lr)
        self._r_full = F.softplus(self.r_raw[None, :, None] + self.dh + kap) + 1e-3
        return lr + delta, delta

    def mix(self, buf):
        if not self.graph:
            return buf
        om = self.omega
        return (1 - om) * buf + om * torch.einsum("ij,...jk->...ik", self.A, buf)

    def kernel(self, tbar):
        m = M.gi_mean_weeks(tbar) if self.gi_temp else torch.full_like(tbar, 2.5)
        return M.gi_kernel(m)                                                  # [B,N,H,K]

    def roll(self, yh, logr, w, sample=0, shift=None):
        """Renewal recursion. yh [B,N,K+1]: last K counts (oldest..newest) and the trailing 52-week total.
        Returns means (sample=0) or samples [S,B,N,H]."""
        yh, cum = yh[..., -1 - M.K_GI:-1], yh[..., -1]
        if sample:
            cum = cum[None].expand(sample, -1, -1).clone()
        pool = torch.exp(self.p)
        r_all = self.disp()
        if sample:
            buf = yh[None].expand(sample, -1, -1, -1).clone()
        else:
            buf = yh.clone()
        outs = []
        om = self.omega
        for h in range(H):
            mix = self.mix(buf)
            lam_in = (mix.flip(-1) * w[..., h, :]).sum(-1)                     # lag 1..K
            lr = logr[..., h] if shift is None else logr[..., h] - shift[..., h]
            if self.deplete:
                lr = lr + torch.log((1 - cum / pool).clamp_min(0.05))
            mu = torch.exp(lr) * lam_in + torch.exp(self.eta)
            r = r_all[..., h]
            nu = self._nu[..., h] if (self.closure and self._nu is not None) else 0.0
            mu_obs = mu * torch.exp(nu) if torch.is_tensor(nu) else mu
            if sample:
                rr = r.expand_as(mu)
                ratio = torch.exp(nu) if torch.is_tensor(nu) else 1.0
                obs = torch.distributions.NegativeBinomial(
                    total_count=rr, probs=(mu_obs / (mu_obs + rr)).clamp(1e-6, 1 - 1e-6)).sample()
                nxt = obs / ratio                     # latent incidence fed back into the renewal
                outs.append(obs)
            else:
                nxt = mu
                outs.append(mu_obs)
            cum = cum + nxt
            buf = torch.cat([buf[..., 1:], nxt[..., None]], -1)
        return torch.stack(outs, -1)

    def forward(self, x, yh, tbar):
        logr, delta = self.log_r(x, yh, tbar)
        w = self.kernel(tbar[..., 3:])
        return self.roll(yh, logr, w), delta

    @torch.no_grad()
    def sample(self, x, yh, tbar, n=200, shift=None, gen=None):
        logr, _ = self.log_r(x, yh, tbar)
        w = self.kernel(tbar[..., 3:])
        return self.roll(yh, logr, w, sample=n, shift=shift)


# ----------------------------------------------------------------------------- neural baselines
class GRUNB(nn.Module):
    """Global recurrent model with negative-binomial likelihood (DeepAR-style likelihood,
    direct multi-horizon head). graph=True mixes hidden states over the coupling graph."""

    def __init__(self, N, A, graph=False):
        super().__init__()
        self.enc = Encoder(N); d = self.enc.out_dim
        self.graph = graph and N > 1
        self.register_buffer("A", torch.tensor(A, dtype=torch.float32))
        if self.graph:
            self.gw = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, 2 * H))

    def forward(self, x, yh, tbar):
        z = self.enc(x)
        if self.graph:
            z = z + F.gelu(self.gw(torch.einsum("ij,bjd->bid", self.A, z)))
        o = self.head(z)
        mu = torch.exp(o[..., :H].clamp(-10, 10) + torch.log1p(yh[..., -2:-1]))
        self._r = F.softplus(o[..., H:]) + 1e-3
        return mu, torch.zeros_like(mu)

    def disp(self):
        return self._r

    @torch.no_grad()
    def sample(self, x, yh, tbar, n=200, shift=None, gen=None):
        mu, _ = self.forward(x, yh, tbar)
        if shift is not None:
            mu = mu * torch.exp(-shift)
        return nb_sample(mu, self._r, n)


class LSTMEnc(nn.Module):
    """LSTM encoder used by the LSTM-GBM hybrid baseline (trained on log1p targets with MSE)."""

    def __init__(self, f_in=9, hid=32):
        super().__init__()
        self.lstm = nn.LSTM(f_in, hid, batch_first=True)
        self.head = nn.Linear(hid, H)

    def forward(self, x):
        B, N, L, Fd = x.shape
        o, (h, _) = self.lstm(x.reshape(B * N, L, Fd))
        h = h[-1]
        return self.head(h).reshape(B, N, H), h.reshape(B, N, -1)


In [ ]:
%%writefile vectra_train.py
"""Training loops and baseline forecasters. Every forecaster returns, for a list of origins,
quantiles Q [O,N,H,len(QS)], means [O,N,H] and (optionally) sample paths [S,O,N,H]."""
import time, numpy as np, torch
from scipy.stats import nbinom
import lightgbm as lgb
from sklearn.linear_model import PoissonRegressor
import vectra_models as VM, vectra_mech as M

H, QS = VM.H, VM.QS
T_START = max(VM.L_HIST, M.K_GI) - 1


def origin_sets(D):
    s = D["split"]
    return (list(range(T_START, s["train_end"])), list(range(s["train_end"], s["val_end"])),
            list(range(s["val_end"], s["end"])))


def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)


def _disp(model, mu):
    r = model.disp()
    return r[None, :, None].expand_as(mu) if r.dim() == 1 else r


def train_torch(model, W, D, seed=0, epochs=250, lr=3e-3, lam_g=0.05, patience=30, bs=16, verbose=False,
                dfl=None, refit=False, fixed_epochs=None):
    """Early stopping on the validation year. With refit=True the caller re-trains a fresh model on
    train+validation origins for the selected number of epochs (fixed_epochs) before testing."""
    set_seed(seed)
    tr, va, _ = origin_sets(D)
    te, ve = D["split"]["train_end"], D["split"]["val_end"]
    if fixed_epochs is not None:
        tr = tr + va; te = ve
        epochs, patience = fixed_epochs, 10 ** 9
    btr = W.batch(tr, te); bva = W.batch(va, ve)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best, best_state, bad = np.inf, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(tr))
        for k in range(0, len(tr), bs):
            idx = perm[k:k + bs]
            x, yh, tb, tg, mk = [a[idx] for a in btr]
            mu, delta = model(x, yh, tb)
            nll = (VM.nb_nll(tg, mu, _disp(model, mu)) * mk).sum() / mk.sum()
            loss = nll + lam_g * (delta ** 2).mean()
            if dfl is not None:
                loss = loss + dfl(model, mu, [tr[i] for i in idx], tb, tg)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            x, yh, tb, tg, mk = bva
            mu, delta = model(x, yh, tb)
            v = ((VM.nb_nll(tg, mu, _disp(model, mu)) * mk).sum() / mk.sum()).item()
            if dfl is not None:
                v = v + dfl(model, mu, va, tb, tg, eval_only=True)
        if fixed_epochs is not None:
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}; best = v
            continue
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose and ep % 20 == 0:
            print(ep, round(v, 4))
    model.load_state_dict(best_state)
    return dict(val_loss=best, epochs=ep + 1, best_epoch=ep + 1 - (bad if fixed_epochs is None else 0),
                train_time=time.time() - t0)


def predict_torch(model, W, origins, end, n=200, seed=0, X=None, y=None, temp=None, shift=None):
    set_seed(seed)
    model.eval()
    x, yh, tb, _, _ = W.batch(origins, end, X=X, y=y, temp=temp)
    with torch.no_grad():
        mu, _ = model(x, yh, tb)
        S = model.sample(x, yh, tb, n=n, shift=shift).numpy()        # [S,O,N,H]
    Q = np.moveaxis(np.quantile(S, QS, axis=0), 0, -1)
    return dict(Q=Q, mean=S.mean(0), S=S.astype(np.float32), tbar=tb.numpy()[..., 3:])


# ----------------------------------------------------------------------------- tabular baselines
def tab_features(W, t, h=None):
    X = W.X
    f = [X[t], X[t - 1][:, :2], X[t - 2][:, :2], X[t - 3][:, :2], X[max(t - 7, 0)][:, :1]]
    return np.concatenate(f, -1)


def _design(W, origins, end, region_onehot=True):
    rows, tg, mk, tb = [], [], [], []
    N = W.y.shape[1]
    for t in origins:
        F_ = tab_features(W, t)
        tbar = W.tbar_future(t)
        if region_onehot:
            F_ = np.concatenate([F_, np.eye(N)], -1)
        rows.append(F_); tb.append(tbar)
        g = np.full((N, H), np.nan)
        for h in range(1, H + 1):
            if t + h <= end:
                g[:, h - 1] = W.y[t + h]
        tg.append(g)
    return np.array(rows), np.array(tg), np.array(tb)


def fit_set(D, refit):
    tr, va, _ = origin_sets(D)
    return (tr + va, D["split"]["val_end"]) if refit else (tr, D["split"]["train_end"])


def persistence(W, D, origins, end, refit=True):
    tr, te = fit_set(D, refit)
    res = {h: [] for h in range(1, H + 1)}
    for t in tr:
        for h in range(1, H + 1):
            if t + h <= te:
                res[h].append(np.log1p(W.y[t + h]) - np.log1p(W.y[t]))
    q = {h: np.quantile(np.concatenate(res[h]), QS) for h in res}
    Q = np.zeros((len(origins), W.y.shape[1], H, len(QS)))
    for o, t in enumerate(origins):
        for h in range(1, H + 1):
            Q[o, :, h - 1] = np.expm1(np.log1p(W.y[t])[:, None] + q[h][None])
    Q = np.clip(Q, 0, None)
    return dict(Q=Q, mean=Q[..., 4])


def glm_nb(W, D, origins, end, refit=True):
    tr, te = fit_set(D, refit)
    Xtr, Ytr, Ttr = _design(W, tr, te)
    Xte, _, Tte = _design(W, origins, end)
    O, N = len(origins), W.y.shape[1]
    Q = np.zeros((O, N, H, len(QS))); MU = np.zeros((O, N, H))
    for h in range(H):
        a = np.concatenate([Xtr, M.log_suitability(Ttr[..., h:h + 1])], -1).reshape(-1, Xtr.shape[-1] + 1)
        yv = Ytr[..., h].reshape(-1); ok = ~np.isnan(yv)
        m = PoissonRegressor(alpha=1e-4, max_iter=1000).fit(a[ok], yv[ok])
        mu_tr = m.predict(a[ok])
        r = np.clip(np.mean(mu_tr ** 2) / max(np.mean((yv[ok] - mu_tr) ** 2 - mu_tr), 1e-6), 0.05, 100)
        b = np.concatenate([Xte, M.log_suitability(Tte[..., h:h + 1])], -1).reshape(-1, Xte.shape[-1] + 1)
        mu = m.predict(b).reshape(O, N)
        MU[:, :, h] = mu
        p = r / (r + mu)
        Q[:, :, h] = nbinom.ppf(QS[None, None], r, p[..., None])
    return dict(Q=Q, mean=MU)


def _lgb_quantiles(Xtr, Ytr, Xte, seed=0):
    O, N = Xte.shape[:2]
    Q = np.zeros((O, N, H, len(QS)))
    for h in range(H):
        a = Xtr[..., h, :] if Xtr.ndim == 4 else Xtr
        yv = np.log1p(Ytr[..., h]).reshape(-1); ok = ~np.isnan(yv)
        A_ = a.reshape(-1, a.shape[-1])[ok]
        B_ = (Xte[..., h, :] if Xte.ndim == 4 else Xte).reshape(-1, A_.shape[-1])
        for k, q in enumerate(QS):
            m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=300, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=20, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, random_state=seed, verbose=-1)
            m.fit(A_, yv[ok])
            Q[:, :, h, k] = np.expm1(m.predict(B_)).reshape(O, N)
    Q = np.sort(np.clip(Q, 0, None), -1)
    return Q


def _with_tbar(X, T):
    s = M.log_suitability(T)                      # [O,N,H]
    return np.concatenate([np.repeat(X[:, :, None, :], H, 2), s[..., None]], -1)


def lightgbm_q(W, D, origins, end, seed=0, refit=True):
    tr, te = fit_set(D, refit)
    Xtr, Ytr, Ttr = _design(W, tr, te)
    Xte, _, Tte = _design(W, origins, end)
    Q = _lgb_quantiles(_with_tbar(Xtr, Ttr), Ytr, _with_tbar(Xte, Tte), seed)
    return dict(Q=Q, mean=Q[..., 4])


def lstm_gbm(W, D, origins, end, seed=0, epochs=200, patience=25, refit=True):
    set_seed(seed)
    tr, va, _ = origin_sets(D)
    te, ve = D["split"]["train_end"], D["split"]["val_end"]
    enc = VM.LSTMEnc()
    opt = torch.optim.Adam(enc.parameters(), lr=3e-3)
    btr, bva = W.batch(tr, te), W.batch(va, ve)
    best, state, bad = np.inf, None, 0
    for ep in range(epochs):
        perm = np.random.permutation(len(tr))
        for k in range(0, len(tr), 16):
            idx = perm[k:k + 16]
            x, _, _, tg, mk = [a[idx] for a in btr]
            p, _ = enc(x)
            loss = (((p - torch.log1p(tg)) ** 2) * mk).sum() / mk.sum()
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            x, _, _, tg, mk = bva
            p, _ = enc(x)
            v = ((((p - torch.log1p(tg)) ** 2) * mk).sum() / mk.sum()).item()
        if v < best - 1e-5:
            best, bad, state = v, 0, {k: t.clone() for k, t in enc.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    enc.load_state_dict(state)

    def hid(orig, e):
        with torch.no_grad():
            x = W.batch(orig, e)[0]
            return enc(x)[1].numpy()
    trf, tef = fit_set(D, refit)
    Xtr, Ytr, Ttr = _design(W, trf, tef); Xte, _, Tte = _design(W, origins, end)
    Xtr = np.concatenate([Xtr, hid(trf, tef)], -1); Xte = np.concatenate([Xte, hid(origins, end)], -1)
    Q = _lgb_quantiles(_with_tbar(Xtr, Ttr), Ytr, _with_tbar(Xte, Tte), seed)
    return dict(Q=Q, mean=Q[..., 4])


In [ ]:
%%writefile vectra_metrics.py
"""Forecast and alarm metrics."""
import numpy as np
from vectra_models import QS

ALPHAS = [(0.025, 0.975), (0.05, 0.95), (0.1, 0.9), (0.25, 0.75)]


def targets(y, origins, end, N, H):
    Y = np.full((len(origins), N, H), np.nan)
    for o, t in enumerate(origins):
        for h in range(1, H + 1):
            if t + h <= end:
                Y[o, :, h - 1] = y[t + h]
    return Y


def wis(Q, Y):
    """Weighted interval score (Bracher et al., 2021) with K=4 central intervals and the median."""
    qi = {q: i for i, q in enumerate(QS)}
    m = Q[..., qi[0.5]]
    s = 0.5 * np.abs(Y - m)
    for lo, hi in ALPHAS:
        a = 2 * lo
        l, u = Q[..., qi[lo]], Q[..., qi[hi]]
        IS = (u - l) + (2 / a) * (l - Y) * (Y < l) + (2 / a) * (Y - u) * (Y > u)
        s = s + (a / 2) * IS
    return s / (len(ALPHAS) + 0.5)


def coverage(lo, hi, Y):
    ok = ~np.isnan(Y)
    return float(((Y >= lo) & (Y <= hi))[ok].mean())


def summary(Q, Y, thr=None, lead=4):
    ok = ~np.isnan(Y)
    qi = {q: i for i, q in enumerate(QS)}
    med = Q[..., qi[0.5]]
    W = wis(Q, Y)
    out = dict(WIS=float(np.nanmean(W)), MAE=float(np.nanmean(np.abs(med - Y))),
               RMSE=float(np.sqrt(np.nanmean((med - Y) ** 2))),
               cov80=coverage(Q[..., qi[0.1]], Q[..., qi[0.9]], Y),
               cov95=coverage(Q[..., qi[0.025]], Q[..., qi[0.975]], Y),
               logMAE=float(np.nanmean(np.abs(np.log1p(med) - np.log1p(Y)))))
    out["WIS_h"] = [float(np.nanmean(W[:, :, h])) for h in range(W.shape[2])]
    out["WIS_region"] = [float(np.nanmean(W[:, i])) for i in range(W.shape[1])]
    if thr is not None:
        a = med[:, :, lead - 1] > thr[None]; tru = Y[:, :, lead - 1] > thr[None]
        v = ~np.isnan(Y[:, :, lead - 1])
        tp = (a & tru & v).sum(); fn = (~a & tru & v).sum(); fp = (a & ~tru & v).sum()
        out.update(alarm_TP=int(tp), alarm_FN=int(fn), alarm_FP=int(fp),
                   alarm_sens=float(tp / max(tp + fn, 1)), alarm_prec=float(tp / max(tp + fp, 1)),
                   alarm_F1=float(2 * tp / max(2 * tp + fp + fn, 1)))
    return out


In [ ]:
%%writefile vectra_decision.py
"""Vector-control allocation: mean LP, CVaR LP (Rockafellar-Uryasev) and SPO+ decision-focused loss."""
import numpy as np, torch
from scipy.optimize import linprog
from scipy.sparse import lil_matrix, csr_matrix
import vectra_mech as M

ACTS = M.ACTIONS
COSTS = np.array([M.COST[a] for a in ACTS])
BUDGET = 6.0
H0 = 3          # interventions act on transmission from week H0 onward


def efficacy(tbar, eff=None, mu0=M.MU0):
    """1 - exp(-phi_a(Tbar)) for each action: array [..., N, H, A]."""
    return np.stack([1 - np.exp(-M.phi(a, tbar, eff, mu0)) for a in ACTS], -1)


CHANNEL = None   # per-region endemic channel; surge control targets the epidemic excess above it


def excess(Y, channel=None):
    ch = CHANNEL if channel is None else channel
    return Y if ch is None else np.maximum(Y - ch[:, None], 0.0)


def averted_matrix(Y, tbar, eff=None, mu0=M.MU0, channel=None):
    """Delta_{i,a} = sum_{h>=H0} (1 - exp(-phi_a(Tbar_{i,h}))) [Y_{i,h} - c_i]^+ .  Y [...,N,H] -> [...,N,A]."""
    E = efficacy(tbar, eff, mu0)
    Z = excess(Y, channel)
    return (E[..., H0 - 1:, :] * Z[..., H0 - 1:, None]).sum(-2)


def _base_constraints(N, budget):
    nA = len(ACTS)
    Aub = lil_matrix((N + 1, N * nA))
    for i in range(N):
        Aub[i, i * nA:(i + 1) * nA] = 1.0
    Aub[N, :] = np.tile(COSTS, N)
    b = np.concatenate([np.ones(N), [budget]])
    return Aub, b


_CACHE = {}


def solve_mean(Delta, budget=BUDGET):
    """max sum x*Delta  s.t. sum_a x_ia <= 1, sum c_a x_ia <= B, 0<=x<=1."""
    N, nA = Delta.shape
    key = (N, budget)
    if key not in _CACHE:
        A_, b_ = _base_constraints(N, budget); _CACHE[key] = (csr_matrix(A_), b_)
    A_, b_ = _CACHE[key]
    r = linprog(-Delta.reshape(-1), A_ub=A_, b_ub=b_, bounds=(0, 1), method="highs")
    return r.x.reshape(N, nA)


def solve_cvar(Ys, Deltas, alpha=0.9, budget=BUDGET):
    """min CVaR_alpha of residual burden  L_s(x) = sum_{i,h>=H0} Y_s - sum x Delta_s."""
    S, N, nA = Deltas.shape
    nx = N * nA
    const = Ys[..., H0 - 1:].sum((-1, -2))                      # [S]
    c = np.concatenate([np.zeros(nx), [1.0], np.full(S, 1.0 / ((1 - alpha) * S))])
    A0, b0 = _base_constraints(N, budget)
    A = lil_matrix((N + 1 + S, nx + 1 + S))
    A[: N + 1, :nx] = A0
    A[N + 1:, :nx] = -Deltas.reshape(S, -1)
    A[N + 1:, nx] = -1.0
    for s in range(S):
        A[N + 1 + s, nx + 1 + s] = -1.0
    b = np.concatenate([b0, -const])
    bounds = [(0, 1)] * nx + [(None, None)] + [(0, None)] * S
    r = linprog(c, A_ub=csr_matrix(A), b_ub=b, bounds=bounds, method="highs")
    return r.x[:nx].reshape(N, nA)


def evaluate(x, Delta_true, budget=BUDGET):
    x_star = solve_mean(Delta_true, budget)
    a, a_star = float((x * Delta_true).sum()), float((x_star * Delta_true).sum())
    return dict(averted=a, oracle=a_star, regret=1 - a / max(a_star, 1e-9))


# ----------------------------------------------------------------------------- SPO+ loss
class SPOPlus:
    """SPO+ surrogate (Elmachtoub & Grigas) for the allocation LP with cost vector c = -Delta / scale.
    Gradient w.r.t. predicted cost is 2 (x*(c) - x*(2c_hat - c))."""

    def __init__(self, W, lam=1.0, budget=BUDGET):
        self.W, self.lam, self.budget = W, lam, budget

    def __call__(self, model, mu, origins, tb, tg, eval_only=False):
        tbf = tb[..., 3:]
        eff = torch.tensor(efficacy(tbf.numpy()), dtype=mu.dtype)                     # [B,N,H,A]
        ch = torch.zeros(mu.shape[1]) if CHANNEL is None else torch.tensor(CHANNEL, dtype=mu.dtype)
        z_hat = torch.relu(mu - ch[None, :, None]); z_true = torch.relu(tg - ch[None, :, None])
        d_hat = (eff[..., H0 - 1:, :] * z_hat[..., H0 - 1:, None]).sum(-2)           # [B,N,A]
        d_true = (eff[..., H0 - 1:, :] * z_true[..., H0 - 1:, None]).sum(-2)
        vals, surr = [], []
        for b in range(mu.shape[0]):
            if (tg[b] == 0).all() or float(d_true[b].sum()) <= 0:
                continue
            scale = float(d_true[b].max(-1).values.sum()) + 1.0
            c = -d_true[b].numpy() / scale
            ch = -d_hat[b] / scale
            x1 = solve_mean(-c, self.budget)
            x2 = solve_mean(-(2 * ch.detach().numpy() - c), self.budget)
            v = -((2 * ch.detach().numpy() - c) * x2).sum() + 2 * (ch.detach().numpy() * x1).sum() - (c * x1).sum()
            vals.append(v)
            surr.append((ch * torch.tensor(2 * (x1 - x2), dtype=mu.dtype)).sum())
        if not vals:
            return 0.0 if eval_only else torch.tensor(0.0)
        if eval_only:
            return self.lam * float(np.mean(vals))
        s = torch.stack(surr).mean()
        return self.lam * (s - s.detach() + float(np.mean(vals)))


In [ ]:
%%writefile vectra_conformal.py
"""Interval calibration: model-native, static split conformal, and adaptive conformal inference
(ACI, Gibbs & Candes 2021) with scores normalised by the model's own predictive spread."""
import numpy as np
from vectra_models import QS

QI = {q: i for i, q in enumerate(QS)}
MAX_HALF = 3.0


def _scale(Q):
    return (np.log1p(Q[..., QI[0.9]]) - np.log1p(Q[..., QI[0.1]])) / 2.5631 + 0.05


def _q(scores, level):
    n = len(scores)
    if n == 0:
        return np.inf
    if level >= 1:
        return np.max(scores) * 2.0
    k = min(n - 1, int(np.ceil((n + 1) * level)) - 1)
    return np.sort(scores)[max(k, 0)]


def calibrate(Q, Y, origins, n_cal, alpha=0.2, mode="aci_region_h", eta=0.05):
    """Q, Y over the concatenated calibration+test origins (chronological). The first n_cal origins
    only provide scores; intervals are returned for the remaining origins.
    mode: raw | split | aci_h | aci_region_h."""
    O, N, H = Y.shape
    m = np.log1p(Q[..., QI[0.5]]); s = _scale(Q)
    score = np.abs(np.log1p(np.nan_to_num(Y, nan=0.0)) - m) / s
    lo = np.zeros((O, N, H)); hi = np.zeros((O, N, H))
    if mode == "raw":
        a = alpha / 2
        ql = min(QS, key=lambda q: abs(q - a)); qh = min(QS, key=lambda q: abs(q - (1 - a)))
        return Q[n_cal:, ..., QI[ql]], Q[n_cal:, ..., QI[qh]]
    keyf = (lambda i, h: (i, h)) if mode == "aci_region_h" else (lambda i, h: h)
    S = {}; A = {}
    # static split: scores from calibration block only, pooled across regions per horizon
    if mode == "split":
        for h in range(H):
            sc = score[:n_cal, :, h][~np.isnan(Y[:n_cal, :, h])]
            qv = _q(sc, 1 - alpha)
            half = np.minimum(qv * s[:, :, h], MAX_HALF)
            lo[:, :, h] = np.expm1(m[:, :, h] - half); hi[:, :, h] = np.expm1(m[:, :, h] + half)
        return np.clip(lo[n_cal:], 0, None), hi[n_cal:]
    t_of = {t: o for o, t in enumerate(origins)}
    for o, t in enumerate(origins):
        # reveal every target whose week is now observed (origin o' with o'+h == t)
        for h in range(1, H + 1):
            op = t_of.get(t - h)
            if op is None:
                continue
            for i in range(N):
                if np.isnan(Y[op, i, h - 1]):
                    continue
                k = keyf(i, h - 1)
                sc = S.setdefault(k, [])
                at = A.setdefault(k, alpha)
                q_used = _q(np.array(sc), 1 - at) if sc else np.inf
                err = float(score[op, i, h - 1] > q_used)
                if o >= n_cal or True:
                    A[k] = at + eta * (alpha - err)
                sc.append(score[op, i, h - 1])
        for i in range(N):
            for h in range(H):
                k = keyf(i, h)
                sc = np.array(S.get(k, []))
                qv = _q(sc, 1 - A.get(k, alpha)) if len(sc) else 3.0
                half = min(qv * s[o, i, h], MAX_HALF)          # cap: at most a factor of e^3 around the median
                lo[o, i, h] = np.expm1(m[o, i, h] - half); hi[o, i, h] = np.expm1(m[o, i, h] + half)
    return np.clip(lo[n_cal:], 0, None), hi[n_cal:]


def interval_metrics(lo, hi, Y, alpha):
    ok = ~np.isnan(Y)
    cov = ((Y >= lo) & (Y <= hi))[ok].mean()
    width = np.log1p(hi)[ok] - np.log1p(lo)[ok]
    IS = (hi - lo) + (2 / alpha) * (lo - Y) * (Y < lo) + (2 / alpha) * (Y - hi) * (Y > hi)
    return dict(coverage=float(cov), log_width=float(np.mean(width)), interval_score=float(np.nanmean(IS[ok])))


In [ ]:
%%writefile vectra_offload.py
"""Deadline- and energy-aware placement of the weekly VECTRA workload over an edge-fog-cloud continuum.

Workloads (CPU cycles) come from timings measured while running the actual pipeline; tier speeds,
power constants, link rates and outage probabilities are scenario assumptions (see DEFAULTS)."""
import numpy as np

DEFAULTS = dict(
    f_edge=(0.6e9, 1.5e9), f_fog=(1.2e9, 3.0e9), f_cloud=(3.0e9, 3.0e9),
    cores_edge=4, cores_fog=8, cores_cloud=32,
    kappa_edge=1.0e-27, kappa_fog=1.5e-27, kappa_cloud=2.0e-27,
    p_tx_edge=0.8,                       # W, edge radio while transmitting
    rate_ef=5e6, rtt_ef=0.04,            # edge<->fog (bit/s, s)
    rate_fc=50e6, rtt_fc=0.08,           # fog<->cloud
    p_out_ef=0.05, deadline=2.0,
)


def dvfs_freq(cycles, cores, slack, fmin, fmax):
    """Lowest frequency that finishes `cycles` (parallel over `cores`) within `slack` seconds."""
    if slack <= 0:
        return None
    f = cycles / (cores * slack)
    if f > fmax:
        return None
    return max(f, fmin)


def task_cost(cycles, cores, f, kappa):
    t = cycles / (cores * f)
    e = kappa * cycles * f ** 2          # dynamic energy  E = kappa * C * f^2
    return t, e


class Simulator:
    def __init__(self, W, n_regions, groups, cfg=None):
        """W: dict of measured workloads in cycles and payload sizes in bits:
        W['infer'] per-region inference (S paths), W['calib'] per-region calibration update,
        W['plan_global'] global allocation LP, W['plan_state'] per-state LP,
        W['in_bits'] per-region input, W['out_bits'] per-region scenario output, W['plan_bits']."""
        self.W, self.N, self.groups = W, n_regions, np.asarray(groups)
        self.c = dict(DEFAULTS, **(cfg or {}))
        self.states = sorted(set(groups))

    def run_week(self, policy, link_fc, link_ef, rng):
        """One weekly cycle. Returns the plan mode actually delivered, energy (J) and latency (s).
        Modes: global / partial (cloud LP, partial = some regions reused last forecasts), state_share (fog LPs
        with last coordinated state shares), state_equal (fog LPs with equal shares), stale (last plan reused),
        reactive (local threshold rule)."""
        c, W = self.c, self.W
        E = 0.0
        ef_up, fc_up, D = link_ef, link_fc, c["deadline"]
        n_up = int(ef_up.sum())
        if policy == "edge":
            t, e = task_cost(W["infer"] + W["calib"], c["cores_edge"], c["f_edge"][1], c["kappa_edge"])
            return dict(ok=t <= D, mode="reactive", energy=self.N * e, latency=t)
        if policy == "cloud":
            E += n_up * c["p_tx_edge"] * W["in_bits"] / c["rate_ef"]
            if not fc_up:
                return dict(ok=False, mode="stale", energy=E, latency=np.nan)
            comm = c["rtt_ef"] + W["in_bits"] / c["rate_ef"] + c["rtt_fc"] + n_up * W["in_bits"] / c["rate_fc"]
            t, e = task_cost(self.N * (W["infer"] + W["calib"]) + W["plan_global"], c["cores_cloud"], c["f_cloud"][1], c["kappa_cloud"])
            L = comm + t + c["rtt_fc"] + W["plan_bits"] / c["rate_fc"] + c["rtt_ef"] + W["plan_bits"] / c["rate_ef"]
            return dict(ok=L <= D, mode=("global" if n_up == self.N else "partial") if L <= D else "stale", energy=E + e, latency=L)
        # fog, greedy and vectra share the regional tier
        if policy == "vectra":
            f = dvfs_freq(W["infer"] + W["calib"], c["cores_edge"], 0.5 * D, *c["f_edge"]) or c["f_edge"][1]
            t_inf, e = task_cost(W["infer"] + W["calib"], c["cores_edge"], f, c["kappa_edge"])
            E += self.N * e + n_up * c["p_tx_edge"] * W["out_bits"] / c["rate_ef"]
            t_ready = t_inf + c["rtt_ef"] + W["out_bits"] / c["rate_ef"]
        else:
            E += n_up * c["p_tx_edge"] * W["in_bits"] / c["rate_ef"]
            per_state = max(np.sum(self.groups == s) for s in self.states)
            t_inf, e = task_cost(per_state * (W["infer"] + W["calib"]), c["cores_fog"], c["f_fog"][1], c["kappa_fog"])
            E += len(self.states) * e
            t_ready = c["rtt_ef"] + W["in_bits"] / c["rate_ef"] + t_inf
        if policy == "fog":
            t_p, e = task_cost(W["plan_state"], c["cores_fog"], c["f_fog"][1], c["kappa_fog"])
            L = t_ready + t_p + c["rtt_ef"]
            return dict(ok=L <= D, mode="state_equal", energy=E + len(self.states) * e, latency=L)
        if fc_up:
            comm = c["rtt_fc"] + n_up * (W["out_bits"] if policy == "vectra" else W["in_bits"]) / c["rate_fc"]
            t_p, e = task_cost(W["plan_global"], c["cores_cloud"], c["f_cloud"][1], c["kappa_cloud"])
            L = t_ready + comm + t_p + c["rtt_fc"] + W["plan_bits"] / c["rate_fc"] + c["rtt_ef"]
            if L <= D:
                return dict(ok=True, mode="global" if n_up == self.N else "partial", energy=E + e, latency=L)
        if policy == "greedy":
            return dict(ok=False, mode="stale", energy=E, latency=np.nan)
        slack = D - t_ready - c["rtt_ef"]
        f = dvfs_freq(W["plan_state"], c["cores_fog"], max(slack, 1e-3), *c["f_fog"]) or c["f_fog"][1]
        t_p, e = task_cost(W["plan_state"], c["cores_fog"], f, c["kappa_fog"])
        L = t_ready + t_p + c["rtt_ef"]
        return dict(ok=L <= D, mode="state_share", energy=E + len(self.states) * e, latency=L)

    def simulate(self, policy, p_out_fc, weeks=52, reps=200, seed=0):
        rng = np.random.default_rng(seed)
        out = []
        for r in range(reps):
            for w in range(weeks):
                fc = rng.random() > p_out_fc
                ef = rng.random(self.N) > self.c["p_out_ef"]
                res = self.run_week(policy, fc, ef, rng)
                out.append(res)
        return out


In [ ]:
%%writefile run_all.py
"""Runs every experiment reported in the VECTRA manuscript and writes results_*.json.
Usage:  python run_all.py [--quick]   (quick: 2 seeds, fewer scenario paths; for Colab smoke tests)"""
import os, sys, json, time, copy, platform, warnings
warnings.filterwarnings("ignore")
import numpy as np, torch
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
import vectra_data as vd, vectra_models as VM, vectra_train as TR, vectra_metrics as MT
import vectra_decision as DC, vectra_conformal as CF, vectra_mech as M, vectra_offload as OF

QUICK = "--quick" in sys.argv
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
NS = 100 if QUICK else 200
OUT = "."
H = VM.H
CFG = dict(vectra=dict(lam_g=0.01, lr=3e-3), grunb=dict(lr=1e-2), graphgru=dict(lr=1e-2), mech=dict(lr=3e-3))


def save(name, obj):
    def conv(o):
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, np.ndarray): return o.tolist()
        raise TypeError(type(o))
    json.dump(obj, open(f"results_{name}.json", "w"), default=conv, indent=1)


def builders(N, A):
    return {
        "VECTRA": lambda: VM.VECTRA(N, A, n_rhat=1),
        "Mechanistic-only": lambda: VM.VECTRA(N, A, n_rhat=1, closure=False),
        "GRU-NB": lambda: VM.GRUNB(N, A),
        "Graph-GRU-NB": lambda: VM.GRUNB(N, A, graph=True),
        # ablations
        "w/o thermal suitability & T-dependent GI": lambda: VM.VECTRA(N, A, n_rhat=1, thermal=False, gi_temp=False),
        "w/o T-dependent GI": lambda: VM.VECTRA(N, A, n_rhat=1, gi_temp=False),
        "w/o coupling graph": lambda: VM.VECTRA(N, A, n_rhat=1, graph=False),
        "w/o observation closure": lambda: VM.VECTRA(N, A, n_rhat=1, obs_closure=False),
    }


def kw_for(name):
    if name.startswith("GRU") or name.startswith("Graph"):
        return CFG["grunb"]
    if name == "Mechanistic-only":
        return CFG["mech"]
    return CFG["vectra"]


CACHE = "."


def fit_pair(mk, W, D, seed, kw, tag=None):
    """Train-only model (early stopped on the validation year) and the refit model (train+validation,
    same number of epochs). Trained weights are cached so that later stages can be re-run."""
    key = None if tag is None else f"cache_{tag.replace('/', '_').replace(' ', '_').replace('&', 'and')}_s{seed}"
    if key and os.path.exists(key + ".json"):
        m0, m1 = mk(), mk()
        m0.load_state_dict(torch.load(key + "_m0.pt")); m1.load_state_dict(torch.load(key + "_m1.pt"))
        return m0, m1, json.load(open(key + ".json"))
    m0 = mk(); info = TR.train_torch(m0, W, D, seed=seed, **kw)          # train-only model (val scores)
    m1 = mk(); info1 = TR.train_torch(m1, W, D, seed=seed, fixed_epochs=info["best_epoch"], **kw)
    inf = dict(best_epoch=info["best_epoch"], train_time=info["train_time"] + info1["train_time"], val_loss=info["val_loss"])
    if key:
        torch.save(m0.state_dict(), key + "_m0.pt"); torch.save(m1.state_dict(), key + "_m1.pt")
        json.dump(inf, open(key + ".json", "w"))
    return m0, m1, inf


def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))


# ============================================================================ main dataset
def main_experiments():
    D = vd.load_meso(); A = vd.coupling_graph(D); W = VM.Window(D, A)
    N = D["y"].shape[1]
    tr, va, te = TR.origin_sets(D); end, ve = D["split"]["end"], D["split"]["val_end"]
    Y = MT.targets(D["y"], te, end, N, H); Yv = MT.targets(D["y"], va, ve, N, H)
    thr = np.percentile(D["y"][: D["split"]["train_end"] + 1], 90, axis=0)
    DC.CHANNEL = thr.copy(); DC.H0 = 3
    dec_or = [t for t in te if t + H <= end]
    Ytrue = MT.targets(D["y"], dec_or, end, N, H)
    Tobs = np.array([np.stack([D["temp"][t + h - 3: t + h + 1].mean(0) for h in range(1, H + 1)], 1) for t in dec_or])
    Dtrue = DC.averted_matrix(Ytrue, Tobs)
    dsel = [te.index(t) for t in dec_or]
    info = dict(dataset=D["name"], N=N, weeks=len(D["y"]), units=D["units"], groups=list(D["groups"]),
                train_end=int(D["split"]["train_end"]), val_end=int(ve), end=int(end),
                climate_missing=D["missing"], n_test_origins=len(te), n_decision_origins=len(dec_or),
                total_cases={int(yr): float(D["y"][D["year"] == yr].sum()) for yr in np.unique(D["year"])},
                threshold_p90=thr, mean_temp=D["temp"][: D["split"]["train_end"] + 1].mean(0),
                omega_graph_degree=int((A > 0).sum(1).mean()))
    save("dataset_info", info)

    def dec_eval(Pmean, tbar, budget=DC.BUDGET, eff=None, mu0=M.MU0, Dt=None):
        Dh = DC.averted_matrix(Pmean, tbar, eff, mu0)
        Dt = Dtrue if Dt is None else Dt
        R = [DC.evaluate(DC.solve_mean(Dh[o], budget), Dt[o], budget) for o in range(len(dec_or))]
        return dict(regret=float(np.mean([r["regret"] for r in R])),
                    regret_p90=float(np.quantile([r["regret"] for r in R], 0.9)),
                    averted=float(sum(r["averted"] for r in R)), oracle=float(sum(r["oracle"] for r in R)),
                    per_origin=[r["regret"] for r in R])

    res = {}; store = {}
    # ---- tabular / statistical baselines
    t0 = time.time(); r = TR.persistence(W, D, te, end); res["Persistence"] = [MT.summary(r["Q"], Y, thr)]
    store["Persistence"] = [r]; res["Persistence"][0]["fit_time"] = time.time() - t0
    t0 = time.time(); r = TR.glm_nb(W, D, te, end); res["NB-GLM"] = [MT.summary(r["Q"], Y, thr)]
    store["NB-GLM"] = [r]; res["NB-GLM"][0]["fit_time"] = time.time() - t0
    res["LightGBM-Q"], store["LightGBM-Q"] = [], []
    res["LSTM-GBM"], store["LSTM-GBM"] = [], []
    for s in SEEDS:
        t0 = time.time(); r = TR.lightgbm_q(W, D, te, end, seed=s); x = MT.summary(r["Q"], Y, thr); x["fit_time"] = time.time() - t0
        res["LightGBM-Q"].append(x); store["LightGBM-Q"].append(r)
        t0 = time.time(); r = TR.lstm_gbm(W, D, te, end, seed=s); x = MT.summary(r["Q"], Y, thr); x["fit_time"] = time.time() - t0
        res["LSTM-GBM"].append(x); store["LSTM-GBM"].append(r)
        print("tabular seed", s, flush=True)
    # also val-period predictions for conformal warm-up of LightGBM (train-only fit)
    lgb_val = TR.lightgbm_q(W, D, va, ve, seed=0, refit=False)

    # ---- neural models and ablations
    B = builders(N, A); val_store = {}; models = {}
    for name, mk in B.items():
        res[name], store[name], val_store[name], models[name] = [], [], [], []
        for s in SEEDS:
            m0, m1, inf = fit_pair(mk, W, D, s, kw_for(name), tag=f"{D['name']}_{name}")
            rv = TR.predict_torch(m0, W, va, ve, n=NS, seed=s)
            t0 = time.time(); r = TR.predict_torch(m1, W, te, end, n=NS, seed=s); inf["predict_time"] = time.time() - t0
            x = MT.summary(r["Q"], Y, thr); x.update(inf); x["params"] = n_params(m1)
            if hasattr(m1, "rho_raw"):
                x.update(gamma=float(m1.gamma), omega=float(m1.omega), rho=float(torch.sigmoid(m1.rho_raw)))
            res[name].append(x); store[name].append(r); val_store[name].append(rv); models[name].append(m1)
            print(name, s, round(x["WIS"], 2), flush=True)
    save("forecast_main", res)

    # ---- decision experiments
    dres = {}
    yt = np.stack([np.repeat(D["y"][t][:, None], H, 1) for t in dec_or])
    tb_dec = store["VECTRA"][0]["tbar"][dsel]
    dres["Reactive (current-week excess)"] = [dec_eval(yt, tb_dec)]
    dres["Persistence"] = [dec_eval(store["Persistence"][0]["Q"][dsel, ..., 4], tb_dec)]
    dres["NB-GLM"] = [dec_eval(store["NB-GLM"][0]["mean"][dsel], tb_dec)]
    for name in ["LightGBM-Q", "LSTM-GBM", "GRU-NB", "Graph-GRU-NB", "Mechanistic-only", "VECTRA"]:
        dres[name] = [dec_eval(r["mean"][dsel], r["tbar"][dsel] if "tbar" in r else tb_dec) for r in store[name]]
    # CVaR variants on VECTRA scenarios
    cv = {}
    for alpha in [0.5, 0.8, 0.9]:
        cv[alpha] = []
        for r in store["VECTRA"]:
            S = r["S"][:, dsel]; Ds = DC.averted_matrix(S, r["tbar"][dsel][None])
            R = [DC.evaluate(DC.solve_cvar(S[:, o], Ds[:, o], alpha), Dtrue[o]) for o in range(len(dec_or))]
            cv[alpha].append(dict(regret=float(np.mean([q["regret"] for q in R])),
                                  regret_p90=float(np.quantile([q["regret"] for q in R], 0.9))))
    dres["_cvar"] = {str(k): v for k, v in cv.items()}
    # regret-trained (SPO+) fine-tuning vs an equal-length likelihood-only fine-tuning
    spo, ctl = [], []
    for s, m in zip(SEEDS, models["VECTRA"]):
        for lst, dfl in [(spo, DC.SPOPlus(W, lam=5.0)), (ctl, None)]:
            m2 = copy.deepcopy(m)
            TR.train_torch(m2, W, D, seed=s, dfl=dfl, fixed_epochs=15, **dict(CFG["vectra"], lr=1e-3))
            r = TR.predict_torch(m2, W, dec_or, end, n=NS, seed=s)
            q = dec_eval(r["mean"], r["tbar"]); rt = TR.predict_torch(m2, W, te, end, n=NS, seed=s)
            q["WIS"] = MT.summary(rt["Q"], Y)["WIS"]; lst.append(q)
    dres["_spo_finetune"] = spo; dres["_likelihood_finetune"] = ctl
    # sensitivity of the allocation advantage to the intervention assumptions
    sens = {}
    base_eff = dict(M.EFF)
    for key, settings in [("budget", [3.0, 6.0, 12.0]), ("efficacy_scale", [0.5, 1.0, 1.5]),
                          ("mu0", [0.07, 0.10, 0.14]), ("channel_pct", [75, 90, 95])]:
        sens[key] = {}
        for v in settings:
            eff = {a: min(0.9, base_eff[a] * (v if key == "efficacy_scale" else 1.0)) for a in base_eff}
            mu0 = v if key == "mu0" else M.MU0; bud = v if key == "budget" else DC.BUDGET
            old = DC.CHANNEL
            if key == "channel_pct":
                DC.CHANNEL = np.percentile(D["y"][: D["split"]["train_end"] + 1], v, axis=0)
            Dt = DC.averted_matrix(Ytrue, Tobs, eff, mu0)
            row = {"Reactive": dec_eval(yt, tb_dec, bud, eff, mu0, Dt)["regret"]}
            for name in ["GRU-NB", "VECTRA", "LightGBM-Q"]:
                row[name] = float(np.mean([dec_eval(r["mean"][dsel], tb_dec, bud, eff, mu0, Dt)["regret"] for r in store[name]]))
            sens[key][str(v)] = row; DC.CHANNEL = old
    dres["_sensitivity"] = sens
    # chosen actions versus temperature (VECTRA seed 0)
    r = store["VECTRA"][0]; Dh = DC.averted_matrix(r["mean"][dsel], r["tbar"][dsel])
    acts = []
    for o in range(len(dec_or)):
        x = DC.solve_mean(Dh[o])
        for i in range(N):
            if x[i].sum() > 1e-6:
                acts.append(dict(temp=float(r["tbar"][dsel][o, i, 2:].mean()), action=int(np.argmax(x[i])), coverage=float(x[i].sum())))
    dres["_actions"] = acts
    # plan quality of each deployment mode (all seeds): global LP, stale plan reuse, fog state LPs with
    # equal shares or with the state shares of the last coordinated plan, and the reactive rule
    states = sorted(set(D["groups"]))
    gidx = {s: np.where(D["groups"] == s)[0] for s in states}
    modes = {k: [] for k in ["global", "stale", "state_equal", "state_share"]}
    for r in store["VECTRA"]:
        Dh = DC.averted_matrix(r["mean"][dsel], r["tbar"][dsel])
        plans = [DC.solve_mean(Dh[o]) for o in range(len(dec_or))]
        g_, st_, se_, ss_ = [], [], [], []
        for o in range(len(dec_or)):
            g_.append(DC.evaluate(plans[o], Dtrue[o])["regret"])
            X = np.zeros_like(Dh[o])
            for s_ in states:
                X[gidx[s_]] = DC.solve_mean(Dh[o][gidx[s_]], DC.BUDGET * len(gidx[s_]) / N)
            se_.append(DC.evaluate(X, Dtrue[o])["regret"])
            if o > 0:
                st_.append(DC.evaluate(plans[o - 1], Dtrue[o])["regret"])
                prev = plans[o - 1]; spent = {s_: float((prev[gidx[s_]] * DC.COSTS).sum()) for s_ in states}
                X = np.zeros_like(Dh[o])
                for s_ in states:
                    if spent[s_] > 1e-9:
                        X[gidx[s_]] = DC.solve_mean(Dh[o][gidx[s_]], spent[s_])
                ss_.append(DC.evaluate(X, Dtrue[o])["regret"])
        modes["global"].append(np.mean(g_)); modes["stale"].append(np.mean(st_))
        modes["state_equal"].append(np.mean(se_)); modes["state_share"].append(np.mean(ss_))
    dres["_deploy_modes"] = {k: float(np.mean(v)) for k, v in modes.items()}
    dres["_deploy_modes"]["reactive"] = dres["Reactive (current-week excess)"][0]["regret"]
    dres["_deploy_modes_sd"] = {k: float(np.std(v)) for k, v in modes.items()}
    save("decision", dres)

    # ---- calibration (val year warms up the online calibrators; train-only models score 2022)
    cres = {}
    for name in ["VECTRA", "GRU-NB", "LightGBM-Q"]:
        cres[name] = {}
        for alpha in [0.2, 0.05]:
            for mode in ["raw", "split", "aci_h", "aci_region_h"]:
                rows = []
                pairs = (zip(val_store[name], store[name]) if name != "LightGBM-Q" else [(lgb_val, store[name][0])])
                for rv, rt in pairs:
                    Qc = np.concatenate([rv["Q"], rt["Q"]]); Yc = np.concatenate([Yv, Y])
                    lo, hi = CF.calibrate(Qc, Yc, va + te, len(va), alpha, mode)
                    mtr = CF.interval_metrics(lo, hi, Y, alpha)
                    cov_reg = [((Y[:, i] >= lo[:, i]) & (Y[:, i] <= hi[:, i]))[~np.isnan(Y[:, i])].mean() for i in range(N)]
                    mtr["region_cov_sd"] = float(np.std(cov_reg)); mtr["region_cov_min"] = float(np.min(cov_reg))
                    # coverage in growth vs decline weeks (sign of the realised 4-week change)
                    g = (Y[:, :, 3] > np.array([D["y"][t] for t in te]))
                    okg = ~np.isnan(Y[:, :, 3])
                    inside = (Y >= lo) & (Y <= hi)
                    mtr["cov_growth_h4"] = float(inside[:, :, 3][g & okg].mean()); mtr["cov_decline_h4"] = float(inside[:, :, 3][~g & okg].mean())
                    mtr["cov_h"] = [float(inside[:, :, h][~np.isnan(Y[:, :, h])].mean()) for h in range(H)]
                    rows.append(mtr)
                cres[name][f"{mode}@{alpha}"] = rows
    eta_s = {}
    for eta in [0.005, 0.01, 0.02, 0.05, 0.1]:
        rows = []
        for rv, rt in zip(val_store["VECTRA"], store["VECTRA"]):
            lo, hi = CF.calibrate(np.concatenate([rv["Q"], rt["Q"]]), np.concatenate([Yv, Y]), va + te, len(va), 0.2, "aci_region_h", eta)
            rows.append(CF.interval_metrics(lo, hi, Y, 0.2))
        eta_s[str(eta)] = rows
    cres["_eta"] = eta_s
    save("calibration", cres)

    # ---- robustness: right truncation of the two most recent weeks, and temperature noise
    rob = {}
    for name in ["VECTRA", "GRU-NB", "Graph-GRU-NB"]:
        rob[name] = {}
        for c in [1.0, 0.8, 0.6]:
            w = []
            for s, m in zip(SEEDS[:3], models[name][:3]):
                Qs = []
                for t in te:
                    Dt_ = dict(D); yy = D["y"].copy(); yy[t] = yy[t] * c; yy[t - 1] = yy[t - 1] * (1 + c) / 2
                    Dt_["y"] = yy; Wt = VM.Window(Dt_, A, feat_stats=W.feat_stats)
                    Qs.append(TR.predict_torch(m, Wt, [t], end, n=NS, seed=s)["Q"])
                w.append(MT.summary(np.concatenate(Qs), Y)["WIS"])
            rob[name][f"trunc_{c}"] = w
        for sd in [1.0, 2.0]:
            w = []
            rng = np.random.default_rng(7)
            Dn = dict(D); tn = D["temp"].copy(); tn[ve - 12:] += rng.normal(0, sd, tn[ve - 12:].shape); Dn["temp"] = tn
            Wn = VM.Window(Dn, A, feat_stats=W.feat_stats)
            for s, m in zip(SEEDS[:3], models[name][:3]):
                w.append(MT.summary(TR.predict_torch(m, Wn, te, end, n=NS, seed=s)["Q"], Y)["WIS"])
            rob[name][f"tnoise_{sd}"] = w
    save("robustness", rob)

    # ---- counterfactual illustration (VECTRA seed 0): intervention-conditioned forecast fans
    m = models["VECTRA"][0]
    cf = {}
    big = np.argsort(-D["y"][te[0]: te[0] + 12].sum(0))[:4]
    t_or = te[5]
    x, yh, tb, _, _ = W.batch([t_or], end)
    for a in ["none"] + list(M.ACTIONS):
        if a == "none":
            shift = None
        else:
            ph = M.phi(a, tb[..., 3:])                                        # torch tensor
            ph[..., : DC.H0 - 1] = 0.0
            shift = ph
        torch.manual_seed(0)
        S = m.sample(x, yh, tb, n=NS, shift=shift).numpy()[:, 0]
        cf[a] = dict(q=np.quantile(S[:, big], [0.1, 0.5, 0.9], axis=0))
    cf["regions"] = [D["units"][i] for i in big]; cf["origin"] = int(t_or)
    cf["history"] = D["y"][t_or - 11: t_or + 1, big].T
    cf["truth"] = D["y"][t_or + 1: t_or + 1 + H, big].T
    save("counterfactual", cf)

    # ---- measured workloads for the deployment simulation
    x, yh, tb, _, _ = W.batch(dec_or[:10], end)
    t0 = time.time()
    for _ in range(3):
        m.sample(x, yh, tb, n=NS)
    t_inf = (time.time() - t0) / (3 * 10 * N)
    r0 = store["VECTRA"][0]; Dh0 = DC.averted_matrix(r0["mean"][dsel], r0["tbar"][dsel])
    t0 = time.time()
    for o in range(len(dec_or)):
        DC.solve_mean(Dh0[o])
    t_plan = (time.time() - t0) / len(dec_or)
    idx = np.where(D["groups"] == "SP")[0]
    t0 = time.time()
    for o in range(len(dec_or)):
        DC.solve_mean(Dh0[o][idx], 2.0)
    t_state = (time.time() - t0) / len(dec_or)
    Qtmp = np.concatenate([val_store["VECTRA"][0]["Q"], store["VECTRA"][0]["Q"]])
    t0 = time.time(); CF.calibrate(Qtmp, np.concatenate([Yv, Y]), va + te, len(va), 0.2, "aci_region_h")
    t_cal = (time.time() - t0) / ((len(va) + len(te)) * N)
    try:
        mhz = float([l for l in open("/proc/cpuinfo") if "cpu MHz" in l][0].split(":")[1])
    except Exception:
        mhz = 2500.0
    f_ref = mhz * 1e6
    wl = dict(infer=t_inf * f_ref, calib=t_cal * f_ref, plan_global=t_plan * f_ref, plan_state=t_state * f_ref,
              in_bits=VM.L_HIST * 9 * 32 + 12 * 32, out_bits=NS * H * 32 + H * 9 * 32, plan_bits=N * 3 * 32,
              f_ref=f_ref, t_infer_s=t_inf, t_plan_s=t_plan, t_state_s=t_state, t_cal_s=t_cal)
    sim = OF.Simulator(wl, N, D["groups"])
    dep = dict(workload=wl, modes_regret=dres["_deploy_modes"], policies={})
    dm = dres["_deploy_modes"]
    mode_reg = {"global": dm["global"], "partial": dm["global"], "state_share": dm["state_share"],
                "state_equal": dm["state_equal"], "stale": dm["stale"], "reactive": dm["reactive"]}
    for pol in ["cloud", "fog", "edge", "greedy", "vectra"]:
        dep["policies"][pol] = {}
        for p in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
            out = sim.simulate(pol, p, reps=50 if QUICK else 200)
            en = np.array([o["energy"] for o in out], float); la = np.array([o["latency"] for o in out], float)
            mm = [o["mode"] for o in out]
            dep["policies"][pol][str(p)] = dict(
                new_plan=float(np.mean([x not in ("stale",) for x in mm])),
                coordinated=float(np.mean([x in ("global", "partial") for x in mm])),
                mode_share={k: float(np.mean([x == k for x in mm])) for k in set(mm)},
                energy_J=float(np.nanmean(en)), latency_p95=float(np.nanpercentile(la, 95)) if np.isfinite(la).any() else None,
                exp_regret=float(np.mean([mode_reg[x] for x in mm])))
    save("deployment", dep)
    return store, models


# ============================================================================ thermal transfer
def transfer_experiment():
    D = vd.load_meso()
    Tm = D["temp"][: D["split"]["train_end"] + 1].mean(0)
    hot = np.where(Tm >= np.median(Tm))[0]; cool = np.where(Tm < np.median(Tm))[0]

    def sub(idx):
        E = dict(D)
        for k in ["y", "temp", "hum", "rain", "delay", "temp_clim", "hum_clim", "rain_clim"]:
            E[k] = D[k][:, idx]
        E["units"] = [D["units"][i] for i in idx]; E["groups"] = D["groups"][idx]; return E
    Dc, Dh = sub(cool), sub(hot)
    Ac, Ah = vd.coupling_graph(Dc), vd.coupling_graph(Dh)
    Wc = VM.Window(Dc, Ac); Wh = VM.Window(Dh, Ah, feat_stats=Wc.feat_stats)
    _, _, te = TR.origin_sets(D); end = D["split"]["end"]
    Yh = MT.targets(Dh["y"], te, end, len(hot), H)

    def move(m, N2, A2):
        m2 = copy.deepcopy(m); sd = m2.state_dict()
        for k, v in list(sd.items()):
            if k != "A" and v.dim() >= 1 and v.shape[0] == len(cool):
                sd[k] = v.mean(0, keepdim=True).expand(N2, *v.shape[1:]).clone()
        if hasattr(m2, "enc"):
            m2.enc.emb = torch.nn.Embedding(N2, sd["enc.emb.weight"].shape[1])
        for nm in ["b", "eta", "r_raw", "p"]:
            if hasattr(m2, nm):
                setattr(m2, nm, torch.nn.Parameter(sd[nm]))
        m2.A = torch.tensor(A2, dtype=torch.float32); sd["A"] = m2.A
        m2.load_state_dict(sd); m2.N = N2
        return m2
    B = builders(len(cool), Ac)
    out = dict(cool_mean_temp=float(Tm[cool].mean()), hot_mean_temp=float(Tm[hot].mean()), n_cool=len(cool), n_hot=len(hot))
    for name in ["VECTRA", "w/o thermal suitability & T-dependent GI", "Mechanistic-only", "GRU-NB", "Graph-GRU-NB"]:
        w = []
        for s in SEEDS:
            _, m1, _ = fit_pair(B[name], Wc, Dc, s, kw_for(name), tag=f"transfer_{name}")
            w.append(MT.summary(TR.predict_torch(move(m1, len(hot), Ah), Wh, te, end, n=NS, seed=s)["Q"], Yh)["WIS"])
        out[name] = w
        print("transfer", name, np.round(w, 2), flush=True)
    save("transfer", out)


# ============================================================================ external dataset
def belem_experiment():
    D = vd.load_belem(); A = np.zeros((1, 1)); W = VM.Window(D, A)
    _, va, te = TR.origin_sets(D); end, ve = D["split"]["end"], D["split"]["val_end"]
    Y = MT.targets(D["y"], te, end, 1, H); Yv = MT.targets(D["y"], va, ve, 1, H)
    thr = np.percentile(D["y"][: D["split"]["train_end"] + 1], 90, axis=0)
    res = {"Persistence": [MT.summary(TR.persistence(W, D, te, end)["Q"], Y, thr)],
           "NB-GLM": [MT.summary(TR.glm_nb(W, D, te, end)["Q"], Y, thr)]}
    res["LightGBM-Q"] = [MT.summary(TR.lightgbm_q(W, D, te, end, seed=s)["Q"], Y, thr) for s in SEEDS]
    B = builders(1, A); cal = {}
    for name in ["VECTRA", "Mechanistic-only", "GRU-NB", "w/o thermal suitability & T-dependent GI"]:
        res[name] = []; cal[name] = []
        for s in SEEDS:
            m0, m1, inf = fit_pair(B[name], W, D, s, kw_for(name), tag=f"{D['name']}_{name}")
            rt = TR.predict_torch(m1, W, te, end, n=NS, seed=s); rv = TR.predict_torch(m0, W, va, ve, n=NS, seed=s)
            x = MT.summary(rt["Q"], Y, thr); x.update(inf); res[name].append(x)
            lo, hi = CF.calibrate(np.concatenate([rv["Q"], rt["Q"]]), np.concatenate([Yv, Y]), va + te, len(va), 0.2, "aci_region_h")
            raw = CF.calibrate(np.concatenate([rv["Q"], rt["Q"]]), np.concatenate([Yv, Y]), va + te, len(va), 0.2, "raw")
            cal[name].append(dict(aci=CF.interval_metrics(lo, hi, Y, 0.2), raw=CF.interval_metrics(*raw, Y, 0.2)))
        print("belem", name, np.round([r["WIS"] for r in res[name]], 2), flush=True)
    res["_calibration"] = cal
    res["_info"] = dict(weeks=len(D["y"]), test_weeks=len(te), train_end=int(D["split"]["train_end"]),
                        years=[int(D["year"][0]), int(D["year"][-1])], mean_temp=float(D["temp"].mean()),
                        test_cases=float(D["y"][D["split"]["val_end"] + 1:].sum()))
    save("belem", res)


if __name__ == "__main__":
    t0 = time.time()
    env = dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__,
               platform=platform.platform(), processor=platform.processor(), cpus=os.cpu_count(), quick=QUICK, seeds=SEEDS, n_paths=NS)
    stages = [a for a in sys.argv[1:] if not a.startswith("--")] or ["main", "transfer", "belem"]
    if "main" in stages: main_experiments()
    if "transfer" in stages: transfer_experiment()
    if "belem" in stages: belem_experiment()
    env["runtime_s"] = time.time() - t0
    save("environment_" + "_".join(stages), env)
    print("done in", round(time.time() - t0), "s")


In [ ]:
%%writefile analyze.py
"""Statistical analysis of the saved results (results_*.json) -> results_analysis.json."""
import json, numpy as np
from scipy.stats import wilcoxon, friedmanchisquare, rankdata

R = lambda n: json.load(open(f"results_{n}.json"))


def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o):
        run = max(run, min(1.0, (m - k) * p[i])); adj[i] = run
    return adj


def rank_biserial(x, y):
    d = np.asarray(x) - np.asarray(y); d = d[d != 0]
    r = rankdata(np.abs(d)); return float((r[d > 0].sum() - r[d < 0].sum()) / r.sum())


def boot_ci(v, B=5000, seed=0, block=4):
    """Moving-block bootstrap of the mean over weekly origins."""
    rng = np.random.default_rng(seed); v = np.asarray(v); n = len(v); out = []
    for _ in range(B):
        idx = np.concatenate([np.arange(s, s + block) % n for s in rng.integers(0, n, int(np.ceil(n / block)))])[:n]
        out.append(v[idx].mean())
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]


def main():
    F = R("forecast_main"); out = {}
    models = [m for m in F if not m.startswith("w/o")]
    summ = {}
    for m, runs in F.items():
        summ[m] = {k: [float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs]))]
                   for k in ["WIS", "MAE", "RMSE", "logMAE", "cov80", "cov95", "alarm_sens", "alarm_prec", "alarm_F1"]}
        summ[m]["WIS_h"] = np.mean([r["WIS_h"] for r in runs], 0).tolist()
        summ[m]["WIS_h_sd"] = np.std([r["WIS_h"] for r in runs], 0).tolist()
        for k in ["params", "train_time", "predict_time", "fit_time", "gamma", "omega", "rho", "best_epoch"]:
            if k in runs[0]:
                summ[m][k] = [float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs]))]
    out["forecast_summary"] = summ
    reg = {m: np.mean([r["WIS_region"] for r in F[m]], 0) for m in F}
    tests = {}
    base = reg["VECTRA"]; ps = []; names = []
    for m in F:
        if m == "VECTRA": continue
        st = wilcoxon(base, reg[m]); ps.append(st.pvalue); names.append(m)
        tests[m] = dict(median_diff=float(np.median(base - reg[m])), p=float(st.pvalue),
                        rank_biserial=rank_biserial(base, reg[m]), vectra_better_regions=int((base < reg[m]).sum()))
    for m, a in zip(names, holm(ps)):
        tests[m]["p_holm"] = float(a)
    out["wilcoxon_vs_vectra_region_WIS"] = tests
    core = ["Persistence", "NB-GLM", "LightGBM-Q", "LSTM-GBM", "GRU-NB", "Graph-GRU-NB", "Mechanistic-only", "VECTRA"]
    M = np.stack([reg[m] for m in core], 1)
    fr = friedmanchisquare(*[M[:, j] for j in range(M.shape[1])])
    ranks = np.mean([rankdata(row) for row in M], 0)
    k, n = M.shape[1], M.shape[0]
    cd = 3.031 * np.sqrt(k * (k + 1) / (6 * n))           # Nemenyi critical difference, q_0.05 for k=8
    out["friedman"] = dict(stat=float(fr.statistic), p=float(fr.pvalue), mean_ranks=dict(zip(core, ranks.tolist())), nemenyi_cd=float(cd))

    Dd = R("decision"); dsum = {}
    for m, runs in Dd.items():
        if m.startswith("_"): continue
        dsum[m] = dict(regret=[float(np.mean([r["regret"] for r in runs])), float(np.std([r["regret"] for r in runs]))],
                       regret_p90=float(np.mean([r["regret_p90"] for r in runs])),
                       averted_frac=float(np.mean([r["averted"] / r["oracle"] for r in runs])))
    out["decision_summary"] = dsum
    per = {m: np.mean([r["per_origin"] for r in Dd[m]], 0) for m in Dd if not m.startswith("_")}
    comp = {}
    for a, b in [("VECTRA", "Reactive (current-week excess)"), ("GRU-NB", "Reactive (current-week excess)"),
                 ("VECTRA", "GRU-NB"), ("VECTRA", "LightGBM-Q"), ("VECTRA", "Mechanistic-only")]:
        d = per[a] - per[b]
        comp[f"{a} - {b}"] = dict(mean=float(d.mean()), ci95=boot_ci(d), wilcoxon_p=float(wilcoxon(per[a], per[b]).pvalue))
    out["decision_paired"] = comp
    sp = Dd["_spo_finetune"]; lk = Dd["_likelihood_finetune"]
    out["spo_vs_likelihood"] = dict(spo_regret=[float(np.mean([r["regret"] for r in sp])), float(np.std([r["regret"] for r in sp]))],
                                    lik_regret=[float(np.mean([r["regret"] for r in lk])), float(np.std([r["regret"] for r in lk]))],
                                    spo_WIS=float(np.mean([r["WIS"] for r in sp])), lik_WIS=float(np.mean([r["WIS"] for r in lk])),
                                    paired_p=float(wilcoxon([r["regret"] for r in sp], [r["regret"] for r in lk]).pvalue))
    out["cvar"] = {a: dict(regret=float(np.mean([r["regret"] for r in v])), regret_p90=float(np.mean([r["regret_p90"] for r in v])))
                   for a, v in Dd["_cvar"].items()}
    C = R("calibration"); cs = {}
    for m, modes in C.items():
        if m.startswith("_"): continue
        cs[m] = {k: {q: float(np.mean([r[q] for r in v])) for q in ["coverage", "log_width", "interval_score", "region_cov_sd", "region_cov_min", "cov_growth_h4", "cov_decline_h4"]}
                 for k, v in modes.items()}
        for k, v in modes.items():
            cs[m][k]["cov_h"] = np.mean([r["cov_h"] for r in v], 0).tolist()
    cs["_eta"] = {e: {q: float(np.mean([r[q] for r in v])) for q in ["coverage", "log_width", "interval_score"]} for e, v in C["_eta"].items()}
    out["calibration_summary"] = cs
    T = R("transfer"); out["transfer_summary"] = {k: ([float(np.mean(v)), float(np.std(v))] if isinstance(v, list) else v) for k, v in T.items()}
    Rb = R("robustness"); out["robustness_summary"] = {m: {k: [float(np.mean(v)), float(np.std(v))] for k, v in d.items()} for m, d in Rb.items()}
    B = R("belem"); bs = {}
    for m, runs in B.items():
        if m.startswith("_"): continue
        bs[m] = {k: [float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs]))] for k in ["WIS", "MAE", "cov80", "cov95", "alarm_F1"]}
    bs["_calibration"] = {m: {k: float(np.mean([r[k]["coverage"] for r in v])) for k in ["raw", "aci"]} for m, v in B["_calibration"].items()}
    bs["_calibration_IS"] = {m: {k: float(np.mean([r[k]["interval_score"] for r in v])) for k in ["raw", "aci"]} for m, v in B["_calibration"].items()}
    bs["_info"] = B["_info"]
    out["belem_summary"] = bs
    json.dump(out, open("results_analysis.json", "w"), indent=1)
    return out


if __name__ == "__main__":
    main(); print("analysis written")


In [ ]:
%%writefile make_schematics.py
"""Schematic figures (Fig. 1 system overview, Fig. 2 forecaster and decision layer).
Black and white, orthogonally routed arrows, no overlapping blocks or labels."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams.update({"font.family": "serif", "font.serif": ["DejaVu Serif"], "font.size": 7,
                     "mathtext.fontset": "dejavuserif"})
DASH = (0, (4, 2.5)); DOT = (0, (1, 1.5))


def box(ax, x0, y0, x1, y1, text, fs=6.6, lw=0.9, fc="white", ls="-"):
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0,rounding_size=0.8",
                                lw=lw, ec="black", fc=fc, ls=ls))
    ax.text((x0 + x1) / 2, (y0 + y1) / 2, text, ha="center", va="center", fontsize=fs, linespacing=1.25)


def route(ax, pts, ls="-", lw=0.9, label=None, seg=0, off=(0, 0), fs=6.0, ha="center"):
    """Polyline with an arrowhead on the last segment; optional label next to segment `seg`."""
    for (x0, y0), (x1, y1) in zip(pts[:-2], pts[1:-1]):
        ax.plot([x0, x1], [y0, y1], color="black", lw=lw, ls=ls, solid_capstyle="butt")
    ax.add_patch(FancyArrowPatch(pts[-2], pts[-1], arrowstyle="-|>", mutation_scale=7, lw=lw, ls=ls,
                                 color="black", shrinkA=0, shrinkB=0))
    if label:
        (x0, y0), (x1, y1) = pts[seg], pts[seg + 1]
        ax.text((x0 + x1) / 2 + off[0], (y0 + y1) / 2 + off[1], label, fontsize=fs, ha=ha, va="center",
                bbox=dict(fc="white", ec="none", pad=0.4))


def tier(ax, x0, y0, x1, y1, title):
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="square,pad=0", lw=0.7, ec="black",
                                fc="#efefef", ls=DASH))
    ax.text(x0 + 0.8, y1 - 0.8, title, fontsize=6.8, fontweight="bold", ha="left", va="top")


def fig1(path):
    fig, ax = plt.subplots(figsize=(7.16, 4.4))
    ax.set_xlim(0, 100); ax.set_ylim(-5, 59); ax.axis("off")
    tier(ax, 18, 1.5, 43, 52, "EDGE (district)")
    tier(ax, 47, 1.5, 70, 52, "FOG (regional)")
    tier(ax, 74, 1.5, 99.5, 52, "CLOUD")
    R1, R2, R3 = (35, 46), (19.5, 30.5), (4, 15)
    box(ax, 0.5, R1[0], 15, R1[1], "Weekly probable\ncases (SINAN)", fs=6.1)
    box(ax, 0.5, R2[0], 15, R2[1], "Station climate\n(INMET): $T$,\nhumidity, rain", fs=6.1)
    box(ax, 0.5, R3[0], 15, R3[1], "Coupling graph\n$\\mathbf{A}$ (training\nco-movement)", fs=6.1)
    box(ax, 20, R1[0], 41, R1[1], "12-week feature\nwindow $\\mathbf{x}_{i,t}$")
    box(ax, 20, R2[0], 41, R2[1], "Thermally propagated\nrenewal forecaster", lw=1.4)
    box(ax, 20, R3[0], 41, R3[1], "Scenario sampler\n($S$ paths, factual and\nintervention-shifted)")
    box(ax, 49, R1[0], 68, R1[1], "Adaptive conformal\ncalibration per\n(region, horizon)")
    box(ax, 49, R2[0], 68, R2[1], "State-level\nallocation LP\n(fallback)")
    box(ax, 49, R3[0], 68, R3[1], "Deadline/DVFS\nplacement and\nplan dispatch")
    box(ax, 76, R1[0], 97.5, R1[1], "Training: NB likelihood\n+ closure penalty\n(weekly refit)")
    box(ax, 76, R2[0], 97.5, R2[1], "Global excess-burden\nallocation LP", lw=1.4)
    box(ax, 76, R3[0], 97.5, R3[1], "Model registry and\nweekly control plan")
    # data -> edge
    route(ax, [(15, 40.5), (20, 40.5)])
    route(ax, [(15, 25), (20, 25)])
    route(ax, [(15, 9.5), (17.3, 9.5), (17.3, 22), (20, 22)])
    route(ax, [(30.5, 35), (30.5, 30.5)])
    route(ax, [(30.5, 19.5), (30.5, 15)])
    # edge -> fog
    route(ax, [(41, 9.5), (45, 9.5), (45, 38), (49, 38)], label="scenarios", seg=1, off=(0, 0), fs=5.8)
    route(ax, [(41, 43), (49, 43)], ls=DASH)
    ax.text(45, 44.4, "$y_{t}$", fontsize=6.2, ha="center", va="bottom")
    route(ax, [(58.5, 35), (58.5, 30.5)], label="calibrated", off=(0, 0), fs=5.8)
    route(ax, [(58.5, 19.5), (58.5, 15)], label="fallback plan", off=(0, 0), fs=5.8)
    # fog <-> cloud
    route(ax, [(68, 43), (76, 43)], ls=DASH)
    ax.text(72, 44.4, "$y_{t+h}$", fontsize=6.2, ha="center", va="bottom")
    route(ax, [(68, 38), (72, 38), (72, 25), (76, 25)], label="$\\Delta$", seg=1, off=(0, 0), fs=6.2)
    route(ax, [(86.75, 19.5), (86.75, 15)], label="plan $\\mathbf{x}^{\\star}$", off=(0, 0), fs=5.8)
    route(ax, [(76, 9.5), (68, 9.5)])
    route(ax, [(49, 6.5), (41, 6.5)])
    ax.text(45, 4.6, "plan", fontsize=5.8, ha="center", va="top")
    # cloud -> edge model weights (above the tiers)
    route(ax, [(86.75, 46), (86.75, 55.5), (38.5, 55.5), (38.5, 46)], label="updated model weights", seg=1,
          off=(0, 0), fs=5.8)
    # legend
    ax.plot([14, 20], [-3, -3], color="black", lw=0.9); ax.text(21, -3, "data and plan flow", va="center", fontsize=6)
    ax.plot([46, 52], [-3, -3], color="black", lw=0.9, ls=DASH); ax.text(53, -3, "outcome feedback", va="center", fontsize=6)
    fig.savefig(path, dpi=600, bbox_inches="tight"); plt.close(fig)


def fig2(path):
    fig, ax = plt.subplots(figsize=(7.16, 4.0))
    ax.set_xlim(0, 100); ax.set_ylim(-1, 57); ax.axis("off")
    R1, R2, R3 = (42, 54), (26, 38), (10, 22)
    A, B, C, Dd, E = (0.5, 16.5), (21, 40), (45, 60), (65, 81), (86, 99.5)
    fs = 6.1
    box(ax, *A[:1], R1[0], A[1], R1[1], "Case history\n$y_{i,t-11:t}$ and\n$(\\mathbf{A}\\mathbf{y})_{i}$", fs=fs)
    box(ax, A[0], R2[0], A[1], R2[1], "Trailing\ntemperature $\\bar{T}_{i,t}$\nand climatology", fs=fs)
    box(ax, A[0], R3[0], A[1], R3[1], "Feature window\n$\\mathbf{x}_{i,t}$ (cases,\nRH, rain, delay)", fs=fs)
    box(ax, B[0], R1[0], B[1], R1[1], "Origin estimate\n$\\hat{R}_{i,t}$ with\nkernel $w(\\bar{T})$", fs=fs)
    box(ax, B[0], R2[0], B[1], R2[1], "Thermal suitability\n$\\log S(\\bar{T})$ and\nEIP $n(\\bar{T})$", fs=fs)
    box(ax, B[0], R3[0], B[1], R3[1], "Bounded GRU\nclosure\n$\\delta_h$, $\\kappa_h$, $\\nu_h$", fs=fs)
    box(ax, C[0], R3[0], C[1], R1[1], "Horizon-wise\n$\\log R_{i,t+h}$\n\n$\\rho^{h}\\log\\hat{R}_{i,t}$\n$+(1-\\rho^{h})b_i$\n$+\\gamma\\,\\Delta\\log S$\n$+\\delta_h$", fs=fs, lw=1.4)
    box(ax, Dd[0], R1[0], Dd[1], R1[1], "Coupled\nrenewal recursion\n$\\lambda_{i,t+h}$ ($\\omega$, $\\mathbf{A}$)", fs=fs)
    box(ax, E[0], R1[0], E[1], R1[1], "NB\nobservation\n($\\nu_h$, $r_{i,h}$)", fs=fs)
    box(ax, Dd[0], R3[0], Dd[1], R3[1], "Intervention\nshift $\\phi_a(\\bar{T})$\non $\\log R$, $h\\geq 3$", fs=fs)
    box(ax, E[0], R3[0], E[1], R3[1], "Averted\nexcess $\\Delta_{i,a}$\nto the LP", fs=fs)
    box(ax, 21, 0, 81, 5.5, "Training loss: NB negative log-likelihood + closure penalty", ls=DOT, fs=fs)
    route(ax, [(A[1], 48), (B[0], 48)])
    route(ax, [(A[1], 32), (B[0], 32)])
    route(ax, [(A[1], 16), (B[0], 16)])
    route(ax, [(8.5, 38), (8.5, 40), (18.7, 40), (18.7, 45), (B[0], 45)])
    route(ax, [(B[1], 48), (C[0], 48)])
    route(ax, [(B[1], 32), (C[0], 32)])
    route(ax, [(B[1], 16), (C[0], 16)])
    route(ax, [(C[1], 48), (Dd[0], 48)])
    route(ax, [(Dd[1], 48), (E[0], 48)])
    route(ax, [(C[1], 16), (Dd[0], 16)])
    route(ax, [(73, 22), (73, 42)], label="counterfactual\nre-simulation", fs=5.6)
    route(ax, [(95.5, 42), (95.5, 22)], label="factual and\nshifted paths", fs=5.6)
    route(ax, [(89, 42), (89, 31), (83.5, 31), (83.5, 2.75), (81, 2.75)])
    ax.text(86.3, 32.6, "$y_{t+h}$", fontsize=6.0, ha="center", va="bottom")
    route(ax, [(30.5, 5.5), (30.5, 10)], ls=DASH)
    ax.text(31.7, 7.7, "$\\nabla_{\\theta}$ (reverse mode)", fontsize=5.8, ha="left", va="center")
    route(ax, [(52.5, 5.5), (52.5, 10)], ls=DASH)
    ax.text(53.7, 7.7, "$\\nabla_{b,\\gamma,\\rho,\\omega}$", fontsize=5.8, ha="left", va="center")
    fig.savefig(path, dpi=600, bbox_inches="tight"); plt.close(fig)


if __name__ == "__main__":
    fig1("fig1_system.png"); fig2("fig2_model.png")


In [ ]:
%%writefile make_figures.py
"""Result figures (Fig. 3 - Fig. 9) from results_*.json and cached models. Grayscale only."""
import json, os, numpy as np, torch, warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vectra_mech as M

plt.rcParams.update({"font.family": "serif", "font.serif": ["DejaVu Serif"], "font.size": 7.5,
                     "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "mathtext.fontset": "dejavuserif", "savefig.dpi": 600})
OUT = "."
R = lambda n: json.load(open(f"results_{n}.json"))
GR = ["black", "0.35", "0.55", "0.75"]
LS = ["-", "--", "-.", ":"]
MK = ["o", "s", "^", "D", "v", "x", "P", "*"]


def panel_label(ax, s):
    ax.text(-0.16, 1.04, s, transform=ax.transAxes, fontsize=8.5, fontweight="bold", va="bottom")


def fig3():
    T = np.linspace(14, 34.3, 400)
    fig, ax = plt.subplots(1, 4, figsize=(7.16, 1.85))
    ax[0].plot(T, np.exp(M.log_suitability(T)), color="black")
    for t in (M.T0, M.TOPT, M.TM):
        ax[0].axvline(t, color="0.6", lw=0.6, ls=":")
    ax[0].set_ylabel("$S(T)$"); ax[0].set_title("Thermal suitability", fontsize=7.5)
    ax[1].plot(T, M.eip_days(T), color="black"); ax[1].scatter([25, 30], [M.EIP25, M.EIP30], s=12, color="black", zorder=3)
    ax[1].set_ylabel("EIP (days)"); ax[1].set_title("Extrinsic incubation", fontsize=7.5)
    ax[2].plot(T, M.gi_mean_weeks(T), color="black"); ax[2].set_ylabel("Mean GI (weeks)"); ax[2].set_title("Generation interval", fontsize=7.5)
    for k, a in enumerate(M.ACTIONS):
        ax[3].plot(T, 1 - np.exp(-M.phi(a, T)), color=GR[k], ls=LS[k], label=a.replace("_", " "))
    ax[3].plot(T, (1 - np.exp(-M.phi("adulticiding", T))) / M.COST["adulticiding"], color="black", lw=0.6, ls=":",
               label="adulticiding per team-week")
    ax[3].set_ylabel("$1-e^{-\\phi_a(T)}$"); ax[3].set_title("Action efficacy", fontsize=7.5)
    ax[3].legend(fontsize=5.4, loc="upper right", handlelength=2.0, borderaxespad=0.1, labelspacing=0.25)
    ax[3].set_ylim(0, 1.32); ax[3].set_yticks([0, 0.25, 0.5, 0.75])
    for i, a in enumerate(ax):
        a.set_xlabel("Temperature (°C)"); panel_label(a, "(" + "abcd"[i] + ")")
    fig.tight_layout(w_pad=1.2); fig.savefig(f"{OUT}/fig3_mechanistic.png"); plt.close(fig)


def fig4():
    A = R("analysis")["forecast_summary"]; F = R("analysis")["friedman"]
    models = ["VECTRA", "GRU-NB", "Graph-GRU-NB", "NB-GLM", "LightGBM-Q", "LSTM-GBM", "Mechanistic-only", "Persistence"]
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.5), gridspec_kw=dict(width_ratios=[1.35, 1]))
    h = np.arange(1, 9)
    for k, m in enumerate(models):
        y = np.array(A[m]["WIS_h"]); e = np.array(A[m]["WIS_h_sd"])
        ax[0].errorbar(h + (k - 3.5) * 0.04, y, yerr=e, color=GR[k % 4], ls=LS[k // 2 % 4], marker=MK[k], ms=3,
                       lw=0.9 if m != "VECTRA" else 1.6, capsize=1.5, label=m)
    ax[0].set_xlabel("Forecast horizon $h$ (weeks)"); ax[0].set_ylabel("WIS (cases)")
    ax[0].legend(fontsize=5.8, ncol=2, loc="upper left", handlelength=2.6, columnspacing=1.0)
    ax[0].set_ylim(0, max(np.array(A["Persistence"]["WIS_h"])) * 1.35)
    mr = F["mean_ranks"]; order = sorted(mr, key=mr.get)
    y = np.arange(len(order))
    ax[1].barh(y, [mr[m] for m in order], color=["black" if m == "VECTRA" else "0.7" for m in order], height=0.6)
    ax[1].set_yticks(y); ax[1].set_yticklabels(order, fontsize=6.4); ax[1].invert_yaxis()
    best = mr[order[0]]
    ax[1].axvline(best + F["nemenyi_cd"], color="black", ls="--", lw=0.7)
    ax[1].text(best + F["nemenyi_cd"] + 0.08, len(order) - 0.6, "best + CD", fontsize=6, va="center")
    ax[1].set_xlabel("Mean rank over 43 regions (lower is better)")
    panel_label(ax[0], "(a)"); panel_label(ax[1], "(b)")
    fig.tight_layout(w_pad=2.0); fig.savefig(f"{OUT}/fig4_forecast.png"); plt.close(fig)


def fig5():
    import vectra_data as vd, vectra_models as VM, vectra_train as TR
    D = vd.load_meso(); A = vd.coupling_graph(D); W = VM.Window(D, A)
    _, _, te = TR.origin_sets(D); end = D["split"]["end"]
    m = VM.VECTRA(43, A, n_rhat=1); m.load_state_dict(torch.load("cache_SE-Brazil_VECTRA_s0_m1.pt"))
    r = TR.predict_torch(m, W, te, end, n=200, seed=0)
    g = VM.GRUNB(43, A); g.load_state_dict(torch.load("cache_SE-Brazil_GRU-NB_s0_m1.pt"))
    rg = TR.predict_torch(g, W, te, end, n=200, seed=0)
    tot = D["y"][te[0]:].sum(0); picks = np.argsort(-tot)[[0, 3, 9]]
    hh = 4
    fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.1))
    for k, i in enumerate(picks):
        wk = np.array([t + hh for t in te if t + hh <= end]); o = np.arange(len(wk))
        Q = r["Q"][o, i, hh - 1]; x = wk - te[0]
        ax[k].fill_between(x, Q[:, 0], Q[:, 8], color="0.88", lw=0, label="95% band")
        ax[k].fill_between(x, Q[:, 2], Q[:, 6], color="0.7", lw=0, label="80% band")
        ax[k].plot(x, Q[:, 4], color="black", lw=1.1, label="VECTRA median")
        ax[k].plot(x, rg["Q"][o, i, hh - 1, 4], color="0.3", lw=0.9, ls="--", label="GRU-NB median")
        ax[k].plot(x, D["y"][wk, i], "o", ms=1.8, color="black", label="observed")
        ax[k].set_title(f"Mesoregion {D['units'][i]}", fontsize=7.2)
        ax[k].set_xlabel("Epidemiological week, 2023")
        panel_label(ax[k], "(" + "abc"[k] + ")")
    ax[0].set_ylabel("Probable cases (4-week-ahead)")
    ax[0].legend(fontsize=5.6, loc="upper right", handlelength=1.8)
    fig.tight_layout(w_pad=1.0); fig.savefig(f"{OUT}/fig5_fans.png"); plt.close(fig)


def fig6():
    A = R("analysis"); dd = A["decision_summary"]; S = R("decision")["_sensitivity"]
    order = ["Reactive (current-week excess)", "Persistence", "NB-GLM", "LightGBM-Q", "LSTM-GBM", "Mechanistic-only",
             "Graph-GRU-NB", "GRU-NB", "VECTRA"]
    lab = {"Reactive (current-week excess)": "Reactive"}
    fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.3), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
    y = np.arange(len(order))
    mu = [dd[m]["regret"][0] * 100 for m in order]; sd = [dd[m]["regret"][1] * 100 for m in order]
    ax[0].barh(y, mu, xerr=sd, color=["black" if m == "VECTRA" else ("0.45" if m.startswith("Reactive") else "0.75") for m in order],
               height=0.6, error_kw=dict(lw=0.7, capsize=1.5))
    ax[0].set_yticks(y); ax[0].set_yticklabels([lab.get(m, m) for m in order], fontsize=6.3); ax[0].invert_yaxis()
    ax[0].set_xlabel("Mean allocation regret (%)")
    for k, (key, xl) in enumerate([("channel_pct", "Endemic channel percentile"), ("budget", "Weekly budget (team-weeks)")]):
        a = ax[k + 1]; xs = list(S[key].keys())
        for j, (m, nm) in enumerate([("Reactive", "Reactive"), ("LightGBM-Q", "LightGBM-Q"), ("GRU-NB", "GRU-NB"), ("VECTRA", "VECTRA")]):
            a.plot(range(len(xs)), [S[key][v][m] * 100 for v in xs], marker=MK[j], ms=3.5, color=GR[j] if m != "VECTRA" else "black",
                   ls=LS[j], lw=1.5 if m == "VECTRA" else 0.9, label=nm)
        a.set_xticks(range(len(xs))); a.set_xticklabels([str(int(float(v))) for v in xs]); a.set_xlabel(xl)
        a.set_ylabel("Regret (%)")
    ax[1].legend(fontsize=5.8, loc="lower right", borderaxespad=0.2); ax[1].set_ylim(6, 37)
    for i in range(3):
        panel_label(ax[i], "(" + "abc"[i] + ")")
    fig.tight_layout(w_pad=1.2); fig.savefig(f"{OUT}/fig6_decision.png"); plt.close(fig)


def fig7():
    C = R("analysis")["calibration_summary"]["VECTRA"]
    modes = [("raw@0.2", "Model-native"), ("split@0.2", "Split conformal"), ("aci_h@0.2", "ACI per horizon"),
             ("aci_region_h@0.2", "ACI per region-horizon")]
    fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.2))
    h = np.arange(1, 9)
    for k, (m, nm) in enumerate(modes):
        ax[0].plot(h, C[m]["cov_h"], marker=MK[k], ms=3, color=GR[k] if k < 3 else "black", ls=LS[k], lw=1.5 if k == 3 else 0.9, label=nm)
    ax[0].axhline(0.8, color="black", lw=0.6, ls=":"); ax[0].set_ylim(0.55, 1.0)
    ax[0].set_xlabel("Horizon $h$"); ax[0].set_ylabel("Empirical coverage (80% target)")
    ax[0].legend(fontsize=5.6, loc="lower left")
    x = np.arange(len(modes)); w = 0.36
    ax[1].bar(x - w / 2, [C[m]["cov_growth_h4"] for m, _ in modes], w, color="black", label="growth weeks")
    ax[1].bar(x + w / 2, [C[m]["cov_decline_h4"] for m, _ in modes], w, color="0.7", label="decline weeks")
    ax[1].axhline(0.8, color="black", lw=0.6, ls=":"); ax[1].set_ylim(0.4, 1.13); ax[1].set_yticks([0.4,0.5,0.6,0.7,0.8,0.9,1.0])
    ax[1].set_xticks(x); ax[1].set_xticklabels(["native", "split", "ACI-h", "ACI-rh"], fontsize=6.4)
    ax[1].set_ylabel("Coverage at $h=4$"); ax[1].legend(fontsize=5.6, loc="upper left", ncol=2, borderaxespad=0.1, columnspacing=1.2)
    ax[2].bar(x, [C[m]["region_cov_sd"] for m, _ in modes], 0.55, color=["0.75", "0.75", "0.75", "black"])
    ax[2].set_xticks(x); ax[2].set_xticklabels(["native", "split", "ACI-h", "ACI-rh"], fontsize=6.4)
    ax[2].set_ylabel("SD of regional coverage")
    for i in range(3):
        panel_label(ax[i], "(" + "abc"[i] + ")")
    fig.tight_layout(w_pad=1.2); fig.savefig(f"{OUT}/fig7_calibration.png"); plt.close(fig)


def fig8():
    cf = R("counterfactual"); acts = R("decision")["_actions"]
    fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.2))
    names = ["none"] + list(M.ACTIONS)
    for k, j in enumerate([0, 1]):
        a = ax[k]; hist = np.array(cf["history"][j]); xh = np.arange(-11, 1); xf = np.arange(1, 9)
        a.plot(xh, hist, color="black", lw=1.0, marker="o", ms=1.8)
        q = np.array(cf["none"]["q"])[:, j]
        a.fill_between(xf, q[0], q[2], color="0.85", lw=0, label="no action, 80% band")
        for n_, nm in enumerate(names):
            qq = np.array(cf[nm]["q"])[:, j]
            a.plot(xf, qq[1], color=GR[n_] if n_ else "black", ls=LS[n_], lw=1.1, label=nm.replace("_", " "))
        a.plot(xf, np.array(cf["truth"][j]), "o", ms=2.2, color="black", mfc="white", label="observed")
        a.axvline(0.5, color="0.5", lw=0.6, ls=":")
        a.set_xlabel("Weeks relative to forecast origin"); a.set_title(f"Mesoregion {cf['regions'][j]}", fontsize=7.2)
        panel_label(a, "(" + "ab"[k] + ")")
    ax[0].set_ylabel("Probable cases"); ax[0].legend(fontsize=5.4, loc="upper left", handlelength=2.0)
    T = np.array([a_["temp"] for a_ in acts]); C_ = np.array([a_["action"] for a_ in acts])
    bins = np.arange(np.floor(T.min()), np.ceil(T.max()) + 1, 1.0)
    bottom = np.zeros(len(bins) - 1)
    for k in range(3):
        c, _ = np.histogram(T[C_ == k], bins)
        ax[2].bar(bins[:-1] + 0.5, c, 0.9, bottom=bottom, color=["black", "0.55", "0.85"][k], edgecolor="black", lw=0.3,
                  label=M.ACTIONS[k].replace("_", " "))
        bottom += c
    ax[2].set_xlabel("Mean forecast temperature (°C)"); ax[2].set_ylabel("Allocated region-weeks")
    ax[2].set_ylim(0, bottom.max() * 1.62); ax[2].legend(fontsize=5.6, loc="upper right", borderaxespad=0.1); panel_label(ax[2], "(c)")
    fig.tight_layout(w_pad=1.0); fig.savefig(f"{OUT}/fig8_counterfactual.png"); plt.close(fig)


def fig9():
    Dp = R("deployment")["policies"]
    names = [("cloud", "Cloud-centric"), ("fog", "Fog-centric"), ("edge", "Edge-only"), ("greedy", "Latency-greedy"), ("vectra", "VECTRA placement")]
    ps = sorted(Dp["cloud"].keys(), key=float)
    fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.2))
    for k, (p, nm) in enumerate(names):
        st = dict(color="black" if p == "vectra" else GR[k % 4], ls=LS[k % 4], marker=MK[k], ms=3, lw=1.5 if p == "vectra" else 0.9, label=nm)
        ax[0].plot([float(x) for x in ps], [Dp[p][x]["new_plan"] for x in ps], **st)
        ax[1].plot([float(x) for x in ps], [Dp[p][x]["exp_regret"] * 100 for x in ps], **st)
    ax[0].set_xlabel("Backhaul outage probability"); ax[0].set_ylabel("Weeks with a fresh plan")
    ax[0].set_ylim(0.4, 1.04)
    ax[1].set_xlabel("Backhaul outage probability"); ax[1].set_ylabel("Expected regret (%)")
    ax[0].legend(fontsize=5.6, loc="lower left")
    x = np.arange(len(names))
    e = [Dp[p]["0.2"]["energy_J"] for p, _ in names]
    ax[2].bar(x, e, 0.6, color=["0.75"] * 4 + ["black"])
    ax[2].set_xticks(x); ax[2].set_xticklabels(["Cloud", "Fog", "Edge", "Greedy", "VECTRA"], fontsize=5.6)
    ax[2].set_ylabel("Energy per weekly cycle (J)")
    for i in range(3):
        panel_label(ax[i], "(" + "abc"[i] + ")")
    fig.tight_layout(w_pad=1.2); fig.savefig(f"{OUT}/fig9_deployment.png"); plt.close(fig)


if __name__ == "__main__":
    import sys
    which = sys.argv[1:] or ["3", "4", "5", "6", "7", "8", "9"]
    for w in which:
        globals()["fig" + w](); print("fig", w)


## 2. Run all experiments
Set `QUICK = True` for a shorter smoke test.

In [ ]:
QUICK = False
flag = '--quick' if QUICK else ''
!python run_all.py main transfer belem $flag

## 3. Statistical analysis

In [ ]:
!python analyze.py
import json, pandas as pd
A = json.load(open('results_analysis.json'))
fs = pd.DataFrame({m: {k: v[0] for k, v in d.items() if isinstance(v, list) and len(v) == 2 and k in ['WIS','MAE','RMSE','cov80','cov95','alarm_F1']} for m, d in A['forecast_summary'].items()}).T
display(fs.round(3))
display(pd.DataFrame(A['wilcoxon_vs_vectra_region_WIS']).T.round(4))
display(pd.DataFrame({m: {'regret': v['regret'][0], 'sd': v['regret'][1], 'averted_frac': v['averted_frac']} for m, v in A['decision_summary'].items()}).T.round(4))

## 4. Figures

In [ ]:
!python make_schematics.py
!python make_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('fig*.png')):
    print(f); display(Image(f, width=900))